# CS313 — Hoàng Khang: bắt đầu với PSM
Chạy từng cell trên Google Colab. Notebook gồm E1, metric, preprocessing PSM V1 và smoke test MTGFlow (chỉ 3 batch debug, không phải train chính thức). Không cần GPU cho E1.

Dữ liệu làm việc ở `/content`; đầu ra được lưu vào một thư mục mới trên Drive sau khi bạn cấp quyền. Không thực thi script từ repo ngoài trong notebook này. Trước khi train, đọc `ROADMAP_HOANG_KHANG.md` và chốt split/scaler với nhóm. Các con số từ dữ liệu giả bên dưới chỉ là unit test, không phải kết quả mô hình.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA khả dụng: True
GPU: Tesla T4


In [ ]:
import sys, platform, subprocess, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
import requests
import torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('numpy/pandas/sklearn/torch:', np.__version__, pd.__version__, sklearn.__version__, torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('E1 và metric test chạy được bằng CPU; chưa cần cài torch cũ theo README.')


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
numpy/pandas/sklearn/torch: 2.4.6 2.2.3 1.6.1 2.14.1+cu130
CUDA: True
GPU: Tesla T4
E1 và metric test chạy được bằng CPU; chưa cần cài torch cũ theo README.


## 1. Thư mục kết quả
Cell tiếp theo xin quyền mount Drive bằng công cụ chính thức của Colab. Nếu không muốn mount, đặt `USE_DRIVE = False`, rồi tự tải thư mục đầu ra về trước khi kết thúc phiên. Không đưa raw data hoặc checkpoint lớn lên Git.

In [ ]:
USE_DRIVE = True
WORK = Path('/content/cs313_hoang_khang')
WORK.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/CS313_Nhom5/hoang_khang')
else:
    OUTPUT_ROOT = WORK / 'outputs'
RUN = OUTPUT_ROOT / ('e1_psm_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RUN.mkdir(parents=True, exist_ok=False)
DATA = WORK / 'data' / 'PSM'
DATA.mkdir(parents=True, exist_ok=True)
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True).stdout
(RUN / 'environment.txt').write_text(sys.version + '\n' + freeze, encoding='utf-8')
print('Kết quả:', RUN)
print('Dữ liệu làm việc:', DATA)


Mounted at /content/drive
Kết quả: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/e1_psm_20261122T164634_090180Z
Dữ liệu làm việc: /content/cs313_hoang_khang/data/PSM


## 2. Lấy PSM từ cache Drive/local hoặc tải lần đầu
File đã có và đúng SHA-256 được dùng lại; chỉ file thiếu mới tải từ nguồn đã ghim.
Không âm thầm ghi đè file khác checksum. Dữ liệu làm việc vẫn có ở `/content`;
bản lưu lâu dài nằm trong `MyDrive/CS313_Nhom5/data/PSM` khi Drive đã mount.

In [ ]:
import hashlib
import requests
import uuid
from pathlib import Path

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_verified(url, destination, expected_sha256):
    """Reuse matching files; never silently replace a different existing file."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        actual = file_sha256(destination)
        if actual != expected_sha256:
            raise RuntimeError(
                f"Checksum khác bản đã ghim: {destination}. "
                "Không tự ghi đè; hãy kiểm tra file/nguồn dữ liệu."
            )
        return "cache_verified"

    partial = destination.with_name(destination.name + ".part_" + uuid.uuid4().hex)
    try:
        with requests.get(url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with partial.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        if file_sha256(partial) != expected_sha256:
            raise RuntimeError(f"Checksum tải về không khớp: {destination.name}")
        # A second writer must not cause silent replacement of another file.
        if destination.exists():
            raise FileExistsError(f"File vừa xuất hiện trong lúc tải: {destination}")
        partial.replace(destination)
    finally:
        if partial.exists():
            partial.unlink()  # only this function's uniquely named temporary file
    return "downloaded_verified"

import json
import shutil
DATASET_REVISION = '2b66e59ee19dac8f6f19fb5d4997f289fdfea357'
PSM_EXPECTED_SHA = {'train.csv': '816e0e2a5356b3b09d356925cd9a0fe1d327e3b350e9336d26ee3e650c72e112', 'test.csv': '566bc50117d13f58793d830c14975c55f627cab63953265d7d6ce2dbcf2b0f80', 'test_label.csv': 'e5b6b0c716a9b3d52e3a1b06761527811f75a218790c37b8a5e806741aeb0676'}
manifest = {"dataset": "thuml/Time-Series-Library", "revision": DATASET_REVISION, "files": {}}
drive_home = Path("/content/drive/MyDrive")
persistent_psm = drive_home / "CS313_Nhom5" / "data" / "PSM" if drive_home.is_dir() else None
for filename, expected_hash in PSM_EXPECTED_SHA.items():
    url = f"https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/{DATASET_REVISION}/PSM/{filename}"
    target = DATA / filename
    cached = persistent_psm / filename if persistent_psm is not None else target
    if target.is_file() and file_sha256(target) == expected_hash and not cached.exists():
        cached.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, cached)
    state = fetch_verified(url, cached, expected_hash)
    if cached != target:
        if target.exists():
            assert file_sha256(target) == expected_hash, f"Không ghi đè file local khác nguồn: {target}"
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(cached, target)
    manifest["files"][filename] = {"url": url, "bytes": target.stat().st_size,
                                  "sha256": file_sha256(target), "cache_status": state}
    print(filename, state)
(RUN / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


train.csv cache_verified
test.csv cache_verified
test_label.csv cache_verified


1058

## 3. E1: kiểm tra dữ liệu thô
Loại cột timestamp đầu tiên đúng như loader DAMR; chưa chuẩn hóa, chưa tách validation và chưa thay NaN/Inf. Mọi quyết định xử lý phải chốt chung. Train được xem là bình thường theo thiết lập benchmark; file train không có nhãn để kiểm chứng điều đó từ dữ liệu này.

In [ ]:
train_df = pd.read_csv(DATA / 'train.csv')
test_df = pd.read_csv(DATA / 'test.csv')
label_df = pd.read_csv(DATA / 'test_label.csv')
print('Cột train:', train_df.columns.tolist())
print('Cột label:', label_df.columns.tolist())
assert 'timestamp' in str(train_df.columns[0]).lower(), 'Cần kiểm tra lại cột đầu trước khi bỏ.'
assert train_df.columns.tolist() == test_df.columns.tolist(), 'Train/test khác cột hoặc thứ tự.'
assert label_df.shape[1] == 2 and 'timestamp' in str(label_df.columns[0]).lower()
assert len(test_df) == len(label_df), 'Nhãn và test không cùng số điểm.'
np.testing.assert_array_equal(test_df.iloc[:, 0].to_numpy(), label_df.iloc[:, 0].to_numpy())
X_train = train_df.iloc[:, 1:].to_numpy(dtype=np.float64)
X_test = test_df.iloc[:, 1:].to_numpy(dtype=np.float64)
raw_y = label_df.iloc[:, 1].to_numpy(dtype=np.float64)
assert np.isfinite(raw_y).all() and set(np.unique(raw_y)).issubset({0, 1})
y_test = raw_y.astype(np.int64)
assert X_train.shape[1] == X_test.shape[1] and X_train.shape[1] > 0
rows = []
for split, x, y in [('train', X_train, None), ('test', X_test, y_test)]:
    rows.append({'dataset': 'PSM', 'split': split, 'n_points': len(x), 'n_features': x.shape[1],
                 'nan_values': int(np.isnan(x).sum()), 'inf_values': int(np.isinf(x).sum()),
                 'anomaly_rate_point': float(y.mean()) if y is not None else np.nan,
                 'label_status': 'provided' if y is not None else 'normal_by_benchmark_protocol'})
stats = pd.DataFrame(rows)
stats.to_csv(RUN / 'dataset_stats.csv', index=False)
print(stats.to_string(index=False))
print('Chưa xử lý NaN/Inf và chưa quyết định scaler.')


Cột train: ['timestamp_(min)', 'feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24']
Cột label: ['timestamp_(min)', 'label']
dataset split  n_points  n_features  nan_values  inf_values  anomaly_rate_point                 label_status
    PSM train    132481          25        4195           0                 NaN normal_by_benchmark_protocol
    PSM  test     87841          25           0           0            0.277558                     provided
Chưa xử lý NaN/Inf và chưa quyết định scaler.


## 4. Nhãn cửa sổ và unit test
Cửa sổ dài 60, bước 10; điểm cuối lưu theo quy ước exclusive (không bao gồm). Chỉ lưu danh sách ID/nhãn, chưa tạo tensor chứa tất cả cửa sổ để tránh tốn RAM. PSM ở đây được xử lý như một chuỗi; MSL/SMD cần kiểm tra ranh giới thực thể riêng.

In [ ]:
def window_labels(labels, window=60, stride=10):
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.isfinite(labels).all() or not np.isin(labels, [0, 1]).all():
        raise ValueError('Nhãn phải là vector 1D hữu hạn chỉ chứa 0/1.')
    if window <= 0 or stride <= 0:
        raise ValueError('window và stride phải dương.')
    starts = np.arange(0, max(0, len(labels) - window + 1), stride, dtype=np.int64)
    sums = np.concatenate(([0], np.cumsum(labels, dtype=np.int64)))
    y = (sums[starts + window] - sums[starts] > 0).astype(np.int64)
    return starts, y

s, w = window_labels([0, 0, 1, 0, 0, 0], window=3, stride=2)
np.testing.assert_array_equal(s, [0, 2])
np.testing.assert_array_equal(w, [1, 1])
assert len(window_labels([0, 0], 3, 1)[0]) == 0
assert len(window_labels([0, 0, 0], 3, 1)[0]) == 1
starts, y_window = window_labels(y_test, 60, 10)
assert len(starts) > 0
index_df = pd.DataFrame({'window_id': np.arange(len(starts)), 'start': starts,
                         'end_exclusive': starts + 60, 'label': y_window})
index_df.to_csv(RUN / 'test_windows_w60_s10.csv', index=False)
print('Test windows:', len(index_df), '| Tỉ lệ bất thường window:', y_window.mean())
print('Số train windows trước khi tách val:', max(0, (len(X_train) - 60) // 10 + 1))
print('PASS: unit test window labels')


Test windows: 8779 | Tỉ lệ bất thường window: 0.31484223715685156
Số train windows trước khi tách val: 13243
PASS: unit test window labels


## 5. Metric chung — bản khởi đầu
AUROC và average precision trên score liên tục. Trường `auprc` bên dưới **được định nghĩa bằng average precision**, cần cả nhóm đồng ý cách gọi này. Không áp dụng point adjustment. F1 chỉ tính nếu có ngưỡng được chọn trước bằng validation, không tìm ngưỡng tốt nhất trên test. Nếu nhãn chỉ có một lớp, AUROC/AUPRC để NaN thay vì tạo số gây hiểu nhầm.

Các test bên dưới dùng dữ liệu giả, không đưa vào CSV kết quả baseline. Khi nối model thật phải kiểm tra cả window ID, không chỉ số lượng score.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def evaluate_scores(labels, scores, threshold=None):
    y = np.asarray(labels)
    score = np.asarray(scores, dtype=np.float64)
    if y.ndim != 1 or score.ndim != 1 or y.shape != score.shape or y.size == 0:
        raise ValueError('labels/scores phải là hai vector 1D cùng độ dài, khác rỗng.')
    if not np.isfinite(y).all() or not np.isfinite(score).all() or not np.isin(y, [0, 1]).all():
        raise ValueError('Nhãn phải 0/1; score/nhãn không được NaN/Inf.')
    both_classes = len(np.unique(y)) == 2
    result = {'auroc': float(roc_auc_score(y, score)) if both_classes else np.nan,
              'auprc': float(average_precision_score(y, score)) if both_classes else np.nan,
              'f1': np.nan, 'flagged_rate': np.nan}
    if threshold is not None:
        if not np.isscalar(threshold) or not np.isfinite(threshold):
            raise ValueError('threshold phải là một số hữu hạn.')
        pred = score >= float(threshold)
        result.update(f1=float(f1_score(y, pred, zero_division=0)), flagged_rate=float(pred.mean()))
    return result

perfect = evaluate_scores([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9], threshold=0.5)
assert perfect['auroc'] == perfect['auprc'] == perfect['f1'] == 1.0
assert perfect['flagged_rate'] == 0.5
assert evaluate_scores([0, 1], [0.5, 0.5])['auroc'] == 0.5
assert np.isnan(evaluate_scores([0, 0], [0.1, 0.2])['auroc'])
for labels, scores in [([0, 1], [0.1]), ([0, 1], [0.1, np.nan]), ([0, 2], [0.1, 0.2])]:
    try:
        evaluate_scores(labels, scores)
    except ValueError:
        pass
    else:
        raise AssertionError('Dữ liệu không hợp lệ phải bị từ chối.')
print('PASS: metric tests (dữ liệu giả, không phải kết quả model)')
(RUN / 'metric_definition.json').write_text(json.dumps({
    'unit': 'window', 'auprc_definition': 'sklearn.average_precision_score',
    'score_direction': 'higher_is_more_anomalous', 'point_adjustment': False,
    'threshold_policy': 'NOT_SELECTED_YET', 'window': 60, 'stride': 10
}, indent=2), encoding='utf-8')


PASS: metric tests (dữ liệu giả, không phải kết quả model)


231

## 6. Tùy chọn: lấy bản code để đọc, chưa thực thi
Cell này clone bốn repo public về một thư mục mới và checkout detached vào commit đã kiểm tra. Không cài requirements và không gọi script huấn luyện. Giữ bản này làm tham chiếu; viết adapter/sửa tương thích trong repo nhóm và lưu diff. Mount Drive cho phép code bạn chạy truy cập Drive, vì vậy hãy đọc code ngoài trước khi thực thi.

In [ ]:
CLONE_REFERENCES = False  # đổi thành True nếu muốn lấy code để đọc
references = {
    'DAMR': ('wyn404/DAMR', 'faaea34d7468ede43a101858722377abca72d956'),
    'MTGFLOW': ('zqhang/MTGFLOW', 'f8cb9ea225fd7e64cfb4556dae3525accac285f5'),
    'GANF': ('EnyanDai/GANF', 'bce38333b109f325a403dae9aff4987ae5bd6e1f'),
    'usad': ('manigalati/usad', 'e25af45c8e1c32783aed94fc7e5ab85effef2b1a'),
}
if CLONE_REFERENCES:
    ref_root = WORK / ('references_' + RUN.name)
    ref_root.mkdir(exist_ok=False)
    for name, (repo, sha) in references.items():
        target = ref_root / name
        subprocess.run(['git', 'clone', f'https://github.com/{repo}.git', str(target)], check=True)
        subprocess.run(['git', '-C', str(target), 'checkout', '--detach', sha], check=True)
        head = subprocess.run(['git', '-C', str(target), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
        assert head == sha
        print(name, head)
(RUN / 'reference_commits.json').write_text(json.dumps(references, indent=2), encoding='utf-8')
print('Hoàn tất E1 PSM. Gửi dataset_stats.csv và data_manifest.json cho nhóm.')
print('Bước sau: chốt protocol; ưu tiên MTGFlow smoke test ở mục 9–11; notebook này chưa train baseline.')
print('Các file đã lưu:', [p.name for p in RUN.iterdir()])


Hoàn tất E1 PSM. Gửi dataset_stats.csv và data_manifest.json cho nhóm.
Bước sau: chốt protocol; ưu tiên MTGFlow smoke test ở mục 9–11; notebook này chưa train baseline.
Các file đã lưu: ['environment.txt', 'data_manifest.json', 'dataset_stats.csv', 'test_windows_w60_s10.csv', 'metric_definition.json', 'reference_commits.json']


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/cs313_hoang_khang/data/PSM")
dst = Path("/content/drive/MyDrive/CS313_Nhom5/data/PSM")

assert Path("/content/drive/MyDrive").is_dir(), "Hãy mount Drive trước."
dst.mkdir(parents=True, exist_ok=True)

for name in ["train.csv", "test.csv", "test_label.csv"]:
    target = dst / name
    if not target.exists():
        shutil.copy2(src / name, target)
    print(name, "→ đã có trên Drive")

train.csv → đã có trên Drive
test.csv → đã có trên Drive
test_label.csv → đã có trên Drive


In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/CS313_Nhom5/data/PSM")

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")
label_df = pd.read_csv(DATA / "test_label.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Label:", label_df.shape)

display(train_df.head())
display(label_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (132481, 26)
Test: (87841, 26)
Label: (87841, 2)


,timestamp_(min),feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24
0,0.0,0.732689,0.761748,0.606848,0.488746,0.424310,0.403609,0.519318,0.398792,0.451453,...,0.318797,0.451856,0.571500,0.469717,0.609883,0.008432,0.0,0.481838,0.006536,0.138249
1,1.0,0.732799,0.761855,0.607133,0.488781,0.432008,0.410256,0.511364,0.402568,0.455657,...,0.321463,0.456123,0.562226,0.466533,0.629812,0.008432,0.0,0.477218,0.006536,0.115207
2,2.0,0.732938,0.761594,0.606895,0.488791,0.418858,0.407724,0.488636,0.396526,0.456104,...,0.347219,0.456692,0.572002,0.487845,0.643598,0.006745,0.0,0.492623,0.008715,0.092166
3,3.0,0.732893,0.761656,0.606478,0.488802,0.417896,0.404242,0.500000,0.405589,0.460020,...,0.361904,0.460532,0.563354,0.479512,0.644690,0.008432,0.0,0.457064,0.008715,0.142857
4,4.0,0.732788,0.761573,0.606777,0.488800,0.421103,0.407407,0.511364,0.399547,0.458507,...,0.359767,0.458825,0.563354,0.448298,0.629948,0.006745,0.0,0.472223,0.006536,0.170507


,timestamp_(min),label
0,132480.0,0
1,132481.0,0
2,132482.0,0
3,132483.0,0
4,132484.0,0


In [ ]:
# Kiểm tra trước khi bỏ cột timestamp
assert "timestamp" in str(train_df.columns[0]).lower()
assert train_df.columns.tolist() == test_df.columns.tolist()
assert label_df.shape[1] == 2
assert len(test_df) == len(label_df)

# Nhãn phải khớp thời điểm với dữ liệu test
np.testing.assert_array_equal(
    test_df.iloc[:, 0].to_numpy(),
    label_df.iloc[:, 0].to_numpy()
)

# Bỏ timestamp, chỉ lấy các biến đầu vào
X_train = train_df.iloc[:, 1:].to_numpy(dtype=np.float64)
X_test = test_df.iloc[:, 1:].to_numpy(dtype=np.float64)

# Lấy cột nhãn
y_test = label_df.iloc[:, 1].to_numpy(dtype=np.float64)
assert np.isfinite(y_test).all()
assert np.isin(y_test, [0, 1]).all()

print("Kiểm tra cấu trúc và nhãn: OK")

Kiểm tra cấu trúc và nhãn: OK


In [ ]:
stats = pd.DataFrame({
    "dataset": ["PSM", "PSM"],
    "split": ["train", "test"],
    "n_points": [len(X_train), len(X_test)],
    "n_features": [X_train.shape[1], X_test.shape[1]],
    "nan_values": [
        int(np.isnan(X_train).sum()),
        int(np.isnan(X_test).sum())
    ],
    "inf_values": [
        int(np.isinf(X_train).sum()),
        int(np.isinf(X_test).sum())
    ],
    "anomaly_points": [pd.NA, int(y_test.sum())],
    "anomaly_rate_pct": [np.nan, float(y_test.mean() * 100)]
})

display(stats)

# Thống kê giá trị thiếu theo từng biến
missing = train_df.iloc[:, 1:].isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
display(missing.rename("nan_count").to_frame())

,dataset,split,n_points,n_features,nan_values,inf_values,anomaly_points,anomaly_rate_pct
0,PSM,train,132481,25,4195,0,<NA>,NaN
1,PSM,test,87841,25,0,0,24381,27.755832


,nan_count
feature_21,2136
feature_6,563
feature_13,249
feature_5,249
feature_22,204
feature_4,202
feature_18,202
feature_14,202
feature_7,47
feature_10,47


In [ ]:
from datetime import datetime

OUT = (
    Path("/content/drive/MyDrive/CS313_Nhom5/hoang_khang")
    / ("E1_PSM_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
)
OUT.mkdir(parents=True, exist_ok=False)

stats.to_csv(OUT / "dataset_stats.csv", index=False)
missing.rename_axis("feature").rename("nan_count").to_csv(
    OUT / "missing_values_by_feature.csv"
)

print("Đã lưu tại:", OUT)

Đã lưu tại: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E1_PSM_20261122_164646_433275


In [ ]:
display(index_df.head(10))
print("File cửa sổ đã lưu tại:", RUN / "test_windows_w60_s10.csv")

,window_id,start,end_exclusive,label
0,0,0,60,0
1,1,10,70,0
2,2,20,80,0
3,3,30,90,1
4,4,40,100,1
5,5,50,110,1
6,6,60,120,1
7,7,70,130,1
8,8,80,140,1
9,9,90,150,1


File cửa sổ đã lưu tại: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/e1_psm_20261122T164634_090180Z/test_windows_w60_s10.csv


## 7. Chuẩn bị dữ liệu cho MTGFlow — V1 tạm


In [ ]:
import random
from sklearn.preprocessing import StandardScaler

PROTOCOL_ID = "DAMR_public_code_PSM_v1"
SEED = 18
WINDOW = 60
STRIDE = 10
VAL_RATIO = 0.2

# Match DataFrame.values slicing and np.nan_to_num from PSMSegLoader.
# np.nan_to_num defaults to copy=True, so the raw DataFrames are preserved.
train_clean = np.nan_to_num(train_df.values[:, 1:])
test_clean = np.nan_to_num(test_df.values[:, 1:])

# Author fits the FULL train file, BEFORE splitting windows into train/val.
scaler = StandardScaler()
scaler.fit(train_clean)
train_z = scaler.transform(train_clean).astype(np.float32)
test_z = scaler.transform(test_clean).astype(np.float32)

assert np.isfinite(train_z).all() and np.isfinite(test_z).all()
# Remove stale temporal-split variables from the old V1 in this runtime.
for old_name in ["split", "train_raw", "val_raw", "val_z"]:
    globals().pop(old_name, None)

print("Protocol:", PROTOCOL_ID)
print("Scaler fit rows:", int(scaler.n_samples_seen_))
print("FULL train:", train_z.shape, "| test:", test_z.shape)
print("Validation will be split by WINDOW IDs in section 8, not by raw rows.")

Protocol: DAMR_public_code_PSM_v1
Scaler fit rows: 132481
FULL train: (132481, 25) | test: (87841, 25)
Validation will be split by WINDOW IDs in section 8, not by raw rows.


In [ ]:
import json
import joblib
from pathlib import Path
from datetime import datetime, timezone

assert Path("/content/drive/MyDrive").is_dir(), "Mount Drive first."
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
SAVE = Path("/content/drive/MyDrive/CS313_Nhom5/hoang_khang") / (
    f"preprocess_DAMR_repro_seed{SEED}_" + stamp
)
SAVE.mkdir(parents=True, exist_ok=False)
config = {
    "protocol_id": PROTOCOL_ID,
    "dataset": "PSM", "seed": SEED,
    "missing_policy": "np.nan_to_num_default_as_DAMR_PSM",
    "scaler": "StandardScaler",
    "scaler_fit_on": "full_train_file_before_window_split",
    "split_policy": "random_start_contiguous_validation_window_block",
    "val_ratio": VAL_RATIO, "window": WINDOW, "stride": STRIDE,
    "train_file_rows": len(train_z), "test_rows": len(test_z),
    "n_features": train_z.shape[1],
    "reference": "DAMR/data_loader.py:PSMSegLoader,get_loader_segment",
}
(SAVE / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
joblib.dump(scaler, SAVE / "scaler.joblib")
print("NEW reproduction output folder:", SAVE)

NEW reproduction output folder: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/preprocess_DAMR_repro_seed18_20261122T172650_900654Z


## 8. Dataset và DataLoader cho MTGFlow — V1 tạm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PSMWindowDataset(Dataset):
    def __init__(self, values, labels=None, window=60, stride=10):
        self.values = np.asarray(values, dtype=np.float32)

        assert self.values.ndim == 2
        assert np.isfinite(self.values).all()
        assert window > 0 and stride > 0
        assert len(self.values) >= window

        self.window = window
        self.starts = np.arange(
            0, len(self.values) - window + 1, stride
        )

        self.labels = None if labels is None else np.asarray(labels)

        if self.labels is not None:
            assert self.labels.shape == (len(self.values),)
            assert np.isin(self.labels, [0, 1]).all()

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, index):
        start = int(self.starts[index])
        end = start + self.window

        # (60, 25) → (25, 60, 1), theo input MTGFlow
        x = torch.from_numpy(self.values[start:end])
        x = x.transpose(0, 1).unsqueeze(-1).contiguous()

        # -1 chỉ là ký hiệu không có nhãn gốc cho train/validation
        if self.labels is None:
            label = -1
        else:
            label = int(np.any(self.labels[start:end] == 1))

        return x, label, index

In [ ]:
from torch.utils.data import Subset

# Keep PSMWindowDataset unchanged: it adapts windows to MTGFlow's input shape.
full_train_dataset = PSMWindowDataset(train_z, window=WINDOW, stride=STRIDE)
test_dataset = PSMWindowDataset(
    test_z, labels=y_test, window=WINDOW, stride=STRIDE
)

# Exact window-ID splitting procedure from DAMR get_loader_segment.
dataset_len = len(full_train_dataset)
train_use_len = int(dataset_len * (1 - VAL_RATIO))
val_use_len = int(dataset_len * VAL_RATIO)
assert train_use_len > 0 and val_use_len > 0
val_start_index = random.Random(SEED).randrange(train_use_len)
all_window_ids = np.arange(dataset_len, dtype=np.int64)
train_window_ids = np.concatenate([
    all_window_ids[:val_start_index],
    all_window_ids[val_start_index + val_use_len:]
])
val_window_ids = all_window_ids[val_start_index:val_start_index + val_use_len]

train_dataset = Subset(full_train_dataset, train_window_ids.tolist())
val_dataset = Subset(full_train_dataset, val_window_ids.tolist())

# Reference DAMR loader settings. Smoke test still has its own small batch 8.
# Final baseline hyperparameters must be documented separately per model.
BATCH_SIZE = 512
torch.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

# Save actual IDs, not only a seed, so other models can use identical splits.
np.savez(SAVE / "window_split.npz", train_ids=train_window_ids,
         val_ids=val_window_ids, val_start_index=val_start_index)
config.update({
    "full_train_windows": dataset_len, "train_windows": len(train_dataset),
    "validation_windows": len(val_dataset), "test_windows": len(test_dataset),
    "val_start_window_id": val_start_index,
    "reference_loader_batch_size": BATCH_SIZE,
    "reference_train_drop_last": True,
})
(SAVE / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Full train windows:", dataset_len)
print("train / validation / test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Validation starts at window ID:", val_start_index)
print("Saved split:", SAVE / "window_split.npz")

Full train windows: 13243
train / validation / test: 10595 2648 8779
Validation starts at window ID: 2969
Saved split: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/preprocess_DAMR_repro_seed18_20261122T172650_900654Z/window_split.npz


In [ ]:
for name, loader in [
    ("train", train_loader), ("validation", val_loader), ("test", test_loader)
]:
    x, labels_batch, window_ids = next(iter(loader))
    assert x.shape == (BATCH_SIZE, train_z.shape[1], WINDOW, 1)
    assert torch.isfinite(x).all()
    print(name, "| windows:", len(loader.dataset), "| batch:", tuple(x.shape))

assert len(np.intersect1d(train_window_ids, val_window_ids)) == 0
np.testing.assert_array_equal(
    np.sort(np.concatenate([train_window_ids, val_window_ids])),
    np.arange(len(full_train_dataset))
)
np.testing.assert_array_equal(test_dataset.starts, index_df["start"].to_numpy())
np.testing.assert_array_equal(
    window_labels(y_test, WINDOW, STRIDE)[1], index_df["label"].to_numpy()
)
print("PASS: window IDs partition correctly; test windows/labels match E1.")
print("NOTE: different train/val WINDOW IDs may still overlap raw points, as in author code.")

train | windows: 10595 | batch: (512, 25, 60, 1)
validation | windows: 2648 | batch: (512, 25, 60, 1)
test | windows: 8779 | batch: (512, 25, 60, 1)
PASS: window IDs partition correctly; test windows/labels match E1.
NOTE: different train/val WINDOW IDs may still overlap raw points, as in author code.


In [ ]:
import joblib

loaded_scaler = joblib.load(SAVE / "scaler.joblib")
assert int(loaded_scaler.n_samples_seen_) == len(X_train)
for raw, expected in [(train_clean, train_z), (test_clean, test_z)]:
    restored = loaded_scaler.transform(raw).astype(np.float32)
    np.testing.assert_allclose(restored, expected, rtol=1e-5, atol=1e-6)
print("PASS: full-train scaler reload matches train/test transformations.")

PASS: full-train scaler reload matches train/test transformations.


## Tổng kết sau mục 1–8 — PSM, cấu hình V1 tạm

Ghi chú này là trạng thái trước phần MTGFlow/E1 bổ sung bên dưới.

### Đã hoàn thành
- Tải PSM, lưu dữ liệu và checksum.
- Thống kê E1, kiểm tra dữ liệu thiếu và nhãn.
- Tạo nhãn cửa sổ; unit test đạt.
- Kiểm thử hàm metric bằng dữ liệu giả.
- Chuẩn bị dữ liệu train/validation/test.
- Tạo DataLoader và kiểm tra shape/thứ tự cửa sổ.
- Lưu và kiểm tra tải lại scaler.

### Cấu hình đang thử, chưa thống nhất nhóm
- NaN: điền 0.
- Train/validation: 80%/20% theo thời gian, trước khi tạo cửa sổ.
- StandardScaler chỉ fit trên train subset.
- Window 60, stride 10; nhãn cửa sổ theo any-anomaly.
- AUPRC hiện tính bằng average precision.
- Batch size 32 và seed 16 mới dùng để kiểm thử DataLoader.
- Cách chia validation khác code DAMR gốc, cần nhóm xác nhận.

### Chưa thực hiện
- Chưa import/chạy model MTGFlow.
- Chưa huấn luyện baseline hoặc có kết quả AUROC thực nghiệm.
- Chưa làm E1 MSL/SMD.

## 9. Import MTGFlow — smoke test, không chạy bảng chính
Chạy phần 1–8 trước. CPU hiện tại đủ cho thử nghiệm nhỏ; không bắt buộc đổi runtime/GPU.
Không gọi `main.py`, không cài đè torch cũ, không tự chạy 5 seed/40 epoch của repo.
Chỉ lấy hai file model và license từ commit đã ghim, cache trên Drive và kiểm tra SHA-256.
Python 3.13 bỏ `cgitb`; loại hai import không dùng (`cgitb`, `turtle`) và đổi import NF
sang package riêng để không xung đột repo khác. Lưu diff, không sửa kiến trúc hay loss.

In [ ]:
import hashlib
import requests
import uuid
from pathlib import Path

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_verified(url, destination, expected_sha256):
    """Reuse matching files; never silently replace a different existing file."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        actual = file_sha256(destination)
        if actual != expected_sha256:
            raise RuntimeError(
                f"Checksum khác bản đã ghim: {destination}. "
                "Không tự ghi đè; hãy kiểm tra file/nguồn dữ liệu."
            )
        return "cache_verified"

    partial = destination.with_name(destination.name + ".part_" + uuid.uuid4().hex)
    try:
        with requests.get(url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with partial.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        if file_sha256(partial) != expected_sha256:
            raise RuntimeError(f"Checksum tải về không khớp: {destination.name}")
        # A second writer must not cause silent replacement of another file.
        if destination.exists():
            raise FileExistsError(f"File vừa xuất hiện trong lúc tải: {destination}")
        partial.replace(destination)
    finally:
        if partial.exists():
            partial.unlink()  # only this function's uniquely named temporary file
    return "downloaded_verified"

import ast
import difflib
import importlib
import json
import platform
import shutil
import sys
import types
from datetime import datetime, timezone

required = ["train_dataset", "test_dataset", "train_z", "test_z",
            "X_train", "X_test", "y_test", "scaler", "SAVE", "WINDOW", "STRIDE", "PROTOCOL_ID", "SEED"]
missing = [name for name in required if name not in globals()]
assert not missing, f"Chạy lại các ô phần 1–8 trước. Thiếu: {missing}"

DRIVE_HOME = Path("/content/drive/MyDrive")
assert DRIVE_HOME.is_dir(), "Hãy chạy ô mount Drive trước."
DAY1_ROOT = DRIVE_HOME / "CS313_Nhom5" / "hoang_khang"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
DEBUG_RUN = DAY1_ROOT / ("debug_MTGFlow_PSM_DAMRrepro_" + stamp)
DEBUG_RUN.mkdir(parents=True, exist_ok=False)

MTG_COMMIT = 'f8cb9ea225fd7e64cfb4556dae3525accac285f5'
SOURCE_REFERENCES = {'MTGFLOW/models/MTGFLOW.py': {'url': 'https://raw.githubusercontent.com/zqhang/MTGFLOW/f8cb9ea225fd7e64cfb4556dae3525accac285f5/models/MTGFLOW.py', 'sha256': 'a24f73293558029010626b7a1faf42f17a6df20874c2c06f632a628889bba5d2', 'bytes': 8131}, 'MTGFLOW/models/NF.py': {'url': 'https://raw.githubusercontent.com/zqhang/MTGFLOW/f8cb9ea225fd7e64cfb4556dae3525accac285f5/models/NF.py', 'sha256': '7fce45864832c38d52e43253dd82e3e5c4dcea69b14ec19cbe5efc28276e3144', 'bytes': 16995}, 'MTGFLOW/LICENSE': {'url': 'https://raw.githubusercontent.com/zqhang/MTGFLOW/f8cb9ea225fd7e64cfb4556dae3525accac285f5/LICENSE', 'sha256': '23a15f9f18973b2376cb281cb3bba6fea79c80c1af79982a48920b40f3a25f4f', 'bytes': 1068}}
MODEL_CACHE = DRIVE_HOME / "CS313_Nhom5" / "source_cache" / ("MTGFLOW_" + MTG_COMMIT)
MODEL_LOCAL = Path("/content/cs313_hoang_khang/external") / ("MTGFLOW_" + MTG_COMMIT[:12]) / "models"
MODEL_LOCAL.mkdir(parents=True, exist_ok=True)

for key, meta in SOURCE_REFERENCES.items():
    relative = key.removeprefix("MTGFLOW/")
    cache_path = MODEL_CACHE / relative
    state = fetch_verified(meta["url"], cache_path, meta["sha256"])
    print(relative, state)

# Remove only unused imports incompatible with headless Python 3.13.
# Use a private package namespace to avoid collisions with another repo's `models`.
original = (MODEL_CACHE / "models/MTGFLOW.py").read_text(encoding="utf-8")
replacements = {
    "from cgitb import reset": "# Compatibility: unused cgitb import removed (Python 3.13).",
    "from turtle import forward, shape": "# Compatibility: unused turtle/Tk import removed (headless runtime).",
    "from models.NF import MAF": "from .NF import MAF  # isolated package; same upstream MAF",
}
patched = original
for old, new in replacements.items():
    assert patched.count(old) == 1, f"Upstream khác dự kiến: {old}"
    patched = patched.replace(old, new, 1)
ast.parse(patched)

def write_reference_or_check(path, content):
    path = Path(path)
    if path.exists():
        assert path.read_text(encoding="utf-8") == content, f"Không ghi đè code đã sửa: {path}"
    else:
        path.write_text(content, encoding="utf-8")

write_reference_or_check(MODEL_LOCAL / "__init__.py", "# Private MTGFlow reference package.\n")
write_reference_or_check(MODEL_LOCAL / "MTGFLOW.py", patched)
write_reference_or_check(MODEL_LOCAL / "NF.py", (MODEL_CACHE / "models/NF.py").read_text(encoding="utf-8"))

patch_text = "".join(difflib.unified_diff(
    original.splitlines(keepends=True), patched.splitlines(keepends=True),
    fromfile="upstream/models/MTGFLOW.py", tofile="compat/models/MTGFLOW.py"
))
(DEBUG_RUN / "compatibility.patch").write_text(patch_text, encoding="utf-8")
(DEBUG_RUN / "source_provenance.json").write_text(json.dumps({
    "repository": "https://github.com/zqhang/MTGFLOW", "commit": MTG_COMMIT,
    "files": SOURCE_REFERENCES, "patch_reason": "unused Python 3.13/headless imports + isolated NF import",
    "architecture_or_loss_changed": False
}, indent=2), encoding="utf-8")

package_name = "_cs313_mtgflow_" + MTG_COMMIT[:12]
for name in list(sys.modules):
    if name == package_name or name.startswith(package_name + "."):
        del sys.modules[name]  # only our private namespace, not other repos' `models`
package = types.ModuleType(package_name)
package.__path__ = [str(MODEL_LOCAL)]
package.__package__ = package_name
sys.modules[package_name] = package
module = importlib.import_module(package_name + ".MTGFLOW")
MTGFLOW_CLASS = module.MTGFLOW

source_out = DEBUG_RUN / "source"
source_out.mkdir()
for name in ["MTGFLOW.py", "NF.py", "__init__.py"]:
    shutil.copy2(MODEL_LOCAL / name, source_out / name)
shutil.copy2(MODEL_CACHE / "LICENSE", source_out / "LICENSE")
print("PASS: import MTGFlow; không chạy main.py hay vòng train của repo.")
print("Thư mục debug:", DEBUG_RUN)

models/MTGFLOW.py cache_verified
models/NF.py cache_verified
LICENSE cache_verified
PASS: import MTGFlow; không chạy main.py hay vòng train của repo.
Thư mục debug: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/debug_MTGFlow_PSM_DAMRrepro_20261122T172701_048267Z


### 9.1. Một batch qua model
Tạo model mới, batch debug 8 cửa sổ để CPU nhẹ. Model trả log-probability cho mỗi cửa sổ;
điểm bất thường là **âm log-probability** (không phải xác suất 0–1).
Chưa tính AUROC, chưa dùng nhãn test để chọn model hay tham số.

In [ ]:
import random
import torch
from torch.utils.data import DataLoader

SMOKE = {
    "run_type": "debug_not_official", "seed": SEED,
    "batch_size": 8, "max_train_batches": 3,
    "learning_rate": 0.002, "weight_decay": 0.0005,
    "clip_grad_value": 1.0,
}
random.seed(SMOKE["seed"])
np.random.seed(SMOKE["seed"])
torch.manual_seed(SMOKE["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SMOKE["seed"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))

MODEL_CONFIG = {
    "n_blocks": 1, "input_size": 1, "hidden_size": 32, "n_hidden": 1,
    "window_size": int(WINDOW), "n_sensor": int(train_z.shape[1]),
    "dropout": 0.0, "model": "MAF", "batch_norm": False,
}
# Model settings follow the inspected main.py defaults; debug batch size is smaller.
smoke_model = MTGFLOW_CLASS(**MODEL_CONFIG).to(device)
smoke_train_loader = DataLoader(
    train_dataset, batch_size=SMOKE["batch_size"], shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(SMOKE["seed"])
)
smoke_probe_loader = DataLoader(
    test_dataset, batch_size=SMOKE["batch_size"], shuffle=False, num_workers=0
)
probe_cpu, probe_labels, probe_ids = next(iter(smoke_probe_loader))
probe_x = probe_cpu.to(device)

smoke_model.eval()
with torch.no_grad():
    initial_log_prob = smoke_model.test(probe_x)
assert initial_log_prob.shape == (len(probe_x),)
assert torch.isfinite(initial_log_prob).all()

print("Device:", device)
print("Input:", tuple(probe_x.shape))
print("Output log_prob:", tuple(initial_log_prob.shape))
print("PASS: một log_prob hữu hạn cho mỗi cửa sổ.")
print("Score bất thường = -log_prob; không phải xác suất 0–1.")

preprocess_path = Path(SAVE) / "config.json"
PREPROCESS_CONFIG = json.loads(preprocess_path.read_text(encoding="utf-8"))
assert PREPROCESS_CONFIG["protocol_id"] == PROTOCOL_ID
assert PREPROCESS_CONFIG["seed"] == SMOKE["seed"]
assert PREPROCESS_CONFIG["window"] == WINDOW and PREPROCESS_CONFIG["stride"] == STRIDE
assert PREPROCESS_CONFIG["train_file_rows"] == len(train_z)
assert PREPROCESS_CONFIG["test_rows"] == len(test_z)
assert PREPROCESS_CONFIG["train_windows"] == len(train_dataset)
assert PREPROCESS_CONFIG["validation_windows"] == len(val_dataset)
loaded_scaler = joblib.load(Path(SAVE) / "scaler.joblib")
for raw, expected in [(train_clean, train_z), (test_clean, test_z)]:
    np.testing.assert_allclose(
        loaded_scaler.transform(raw).astype(np.float32), expected, rtol=1e-5, atol=1e-6
    )
for filename in ["config.json", "scaler.joblib", "window_split.npz"]:
    destination_name = "preprocess_config.json" if filename == "config.json" else filename
    shutil.copy2(Path(SAVE) / filename, DEBUG_RUN / destination_name)
env = {
    "python": sys.version, "platform": platform.platform(), "torch": str(torch.__version__),
    "numpy": np.__version__, "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
}
(DEBUG_RUN / "debug_config.json").write_text(json.dumps({
    "smoke": SMOKE, "model": MODEL_CONFIG, "preprocessing": PREPROCESS_CONFIG,
    "environment": env, "official_results": False,
    "config_source": f"zqhang/MTGFLOW@{MTG_COMMIT}; PSM preprocessing/split follows DAMR public code"
}, indent=2, ensure_ascii=False), encoding="utf-8")

Device: cuda:0
Input: (8, 25, 60, 1)
Output log_prob: (8,)
PASS: một log_prob hữu hạn cho mỗi cửa sổ.
Score bất thường = -log_prob; không phải xác suất 0–1.


1584

## 10. Train đúng 3 batch debug
Vòng lặp bị giới hạn bằng `islice(..., 3)`, không train cả dataset/epoch.
Nhãn `-1` của train chỉ báo không có nhãn gốc, hoàn toàn không tham gia loss.
Kiểm tra loss/gradient/trọng số hữu hạn và optimizer thật sự cập nhật tham số.
Ba batch không đủ kết luận model tốt/xấu; loss không bắt buộc giảm đều.

In [ ]:
from itertools import islice
from time import perf_counter

# Bounded by three optimizer steps. No loop over all epochs or all dataset batches.
optimizer = torch.optim.Adam(
    smoke_model.parameters(), lr=SMOKE["learning_rate"],
    weight_decay=SMOKE["weight_decay"]
)
parameters_before = {name: p.detach().cpu().clone() for name, p in smoke_model.named_parameters()}
training_rows = []
started = perf_counter()

for step, (batch_x, ignored_labels, window_ids) in enumerate(
    islice(smoke_train_loader, SMOKE["max_train_batches"]), start=1
):
    smoke_model.train()
    batch_x = batch_x.to(device)
    optimizer.zero_grad(set_to_none=True)
    loss = -smoke_model(batch_x)  # negative mean log-probability; labels are NOT used
    assert loss.ndim == 0 and torch.isfinite(loss), f"Loss không hợp lệ ở step {step}"
    loss.backward()
    for name, parameter in smoke_model.named_parameters():
        if parameter.grad is not None:
            assert torch.isfinite(parameter.grad).all(), f"Gradient không hữu hạn: {name}"
    torch.nn.utils.clip_grad_value_(smoke_model.parameters(), SMOKE["clip_grad_value"])
    optimizer.step()
    for name, parameter in smoke_model.named_parameters():
        assert torch.isfinite(parameter).all(), f"Trọng số không hữu hạn: {name}"
    row = {"run_type": "debug", "step": step, "batch_size": len(batch_x),
           "negative_log_likelihood": float(loss.detach().cpu()), "elapsed_s": perf_counter() - started}
    training_rows.append(row)
    print(f"Step {step}/{SMOKE['max_train_batches']}: loss={row['negative_log_likelihood']:.6f}")

assert len(training_rows) == SMOKE["max_train_batches"]
assert any(not torch.equal(parameters_before[name], p.detach().cpu())
           for name, p in smoke_model.named_parameters()), "Không có trọng số nào được cập nhật."
pd.DataFrame(training_rows).to_csv(DEBUG_RUN / "debug_train_steps.csv", index=False)
with (DEBUG_RUN / "debug_train.log").open("w", encoding="utf-8") as handle:
    for row in training_rows:
        handle.write(json.dumps(row) + "\n")
print("PASS: loss/gradient/trọng số hữu hạn, optimizer có cập nhật.")
print("Chỉ là debug 3 batch; loss không bắt buộc giảm đều. Không phải kết quả paper.")


Step 1/3: loss=1.713377
Step 2/3: loss=2.208989
Step 3/3: loss=1.881478
PASS: loss/gradient/trọng số hữu hạn, optimizer có cập nhật.
Chỉ là debug 3 batch; loss không bắt buộc giảm đều. Không phải kết quả paper.


## 11. Lưu/load checkpoint và đối chiếu score
Lưu state dict + cấu hình, tạo model mới rồi load và so score ở chế độ eval.
Đây là **checkpoint debug để kiểm tra inference**, không phải checkpoint E5 hay bản resume training hoàn chỉnh.
Chỉ có score một batch nhỏ; không tính metric đại diện cho toàn bộ test.

In [ ]:
smoke_model.eval()
with torch.no_grad():
    scores_before = -smoke_model.test(probe_x)
assert torch.isfinite(scores_before).all()

checkpoint_path = DEBUG_RUN / "debug_checkpoint.pt"
torch.save({
    "checkpoint_type": "inference_debug_only_not_training_resume",
    "model_state_dict": smoke_model.state_dict(),
    "model_config": MODEL_CONFIG, "smoke_config": SMOKE,
    "upstream_commit": MTG_COMMIT,
}, checkpoint_path)

# Only load our own checkpoint, containing tensors and primitive metadata.
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored_model = MTGFLOW_CLASS(**checkpoint["model_config"]).to(device)
restored_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
restored_model.eval()
with torch.no_grad():
    scores_after = -restored_model.test(probe_x)
torch.testing.assert_close(scores_after, scores_before, rtol=1e-5, atol=1e-6)

probe_ids_np = probe_ids.numpy()
probe_frame = pd.DataFrame({
    "run_type": "debug", "window_id": probe_ids_np,
    "start": test_dataset.starts[probe_ids_np],
    "end_exclusive": test_dataset.starts[probe_ids_np] + WINDOW,
    "label": probe_labels.numpy(), "score": scores_after.cpu().numpy(),
})
probe_frame.to_csv(DEBUG_RUN / "debug_probe_scores.csv", index=False)
summary = {
    "status": "passed", "run_type": "debug_not_official", "dataset": "PSM",
    "train_steps": len(training_rows), "probe_windows": len(probe_frame),
    "forward_finite": True, "gradients_finite": True, "parameters_updated": True,
    "checkpoint_reload_matches": True, "scaler_reload_matches": True,
    "auroc_computed": False, "full_training_completed": False,
    "note": "Không có metric trên toàn bộ test; không gộp vào bảng E3a.",
}
(DEBUG_RUN / "debug_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("PASS: checkpoint tải lại và tái tạo đúng score trên batch kiểm thử.")
print("Không tính AUROC của batch nhỏ, không ghi vào results.csv chính thức.")
print("Debug artifacts:", DEBUG_RUN)
display(probe_frame.head())


PASS: checkpoint tải lại và tái tạo đúng score trên batch kiểm thử.
Không tính AUROC của batch nhỏ, không ghi vào results.csv chính thức.
Debug artifacts: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/debug_MTGFlow_PSM_DAMRrepro_20261122T172701_048267Z


,run_type,window_id,start,end_exclusive,label,score
0,debug,0,0,60,0,1.592286
1,debug,1,10,70,0,1.586687
2,debug,2,20,80,0,1.580597
3,debug,3,30,90,1,1.578489
4,debug,4,40,100,1,1.566423


## 12. E1 MSL và SMD — dữ liệu gốc, CPU
Tải lần đầu tổng khoảng 277 MB vào `MyDrive/CS313_Nhom5/data/MSL` và `.../SMD`;
lần sau dùng cache có checksum, không tải lại. Không cần file `.pkl`, chỉ dùng `.npy` với `allow_pickle=False`.
Bước này chỉ thống kê, không điền thiếu/chuẩn hóa/chia validation hay train MSL/SMD.
Mảng gộp không kèm ranh giới thực thể: chưa tự tạo window qua các ranh giới chưa biết.

In [ ]:
DATASET_REVISION = '2b66e59ee19dac8f6f19fb5d4997f289fdfea357'
EXTRA_DATA_REFERENCES = {'MSL/MSL_train.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/MSL/MSL_train.npy', 'sha256': 'd767ecfc8ea344f7a6e78f815a84dc7923f600df1beb14359f3e73202148b058', 'bytes': 25659608}, 'MSL/MSL_test.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/MSL/MSL_test.npy', 'sha256': '8b479ebeeabb63ff636e8adeb095a7bcc46a0cdc7befa189b45b05b1e90c9425', 'bytes': 32440888}, 'MSL/MSL_test_label.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/MSL/MSL_test_label.npy', 'sha256': '23e120c30a645000f093ff5311de177601b8220ba3ca4281c6376883444f7b97', 'bytes': 73857}, 'SMD/SMD_train.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/SMD/SMD_train.npy', 'sha256': 'ef3a4a4e9c87c939c23e0cdf1190c18a1d41ef1aadb5d6c5ae909514332ef791', 'bytes': 107677688}, 'SMD/SMD_test.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/SMD/SMD_test.npy', 'sha256': '64520f6b9351d609bb395e947a3c98f6c513f608c794c571f9a0176d98cdb671', 'bytes': 107679968}, 'SMD/SMD_test_label.npy': {'url': 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/SMD/SMD_test_label.npy', 'sha256': '400a29e062ca8c8f8d7e4801c68c35801f3a2180d5d70fe118d6525a33dd3ed2', 'bytes': 2833808}}
EXTRA_DATA_CACHE = DRIVE_HOME / "CS313_Nhom5" / "data"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
E1_EXTRA_RUN = DAY1_ROOT / ("E1_all_public_" + stamp)
E1_EXTRA_RUN.mkdir(parents=True, exist_ok=False)
extra_manifest = {"dataset_source": "thuml/Time-Series-Library", "revision": DATASET_REVISION, "files": {}}

for key, meta in EXTRA_DATA_REFERENCES.items():
    path = EXTRA_DATA_CACHE / key
    state = fetch_verified(meta["url"], path, meta["sha256"])
    extra_manifest["files"][key] = {
        **meta, "path": str(path), "cache_status": state,
        "observed_bytes": path.stat().st_size,
    }
    print(key, state, f"({path.stat().st_size / 1e6:.1f} MB)")
(E1_EXTRA_RUN / "data_manifest_MSL_SMD.json").write_text(
    json.dumps(extra_manifest, indent=2), encoding="utf-8"
)
print("Dữ liệu gốc đã lưu trên Drive. Lần sau kiểm tra checksum rồi dùng lại, không tải lại.")


MSL/MSL_train.npy cache_verified (25.7 MB)
MSL/MSL_test.npy cache_verified (32.4 MB)
MSL/MSL_test_label.npy cache_verified (0.1 MB)
SMD/SMD_train.npy cache_verified (107.7 MB)
SMD/SMD_test.npy cache_verified (107.7 MB)
SMD/SMD_test_label.npy cache_verified (2.8 MB)
Dữ liệu gốc đã lưu trên Drive. Lần sau kiểm tra checksum rồi dùng lại, không tải lại.


In [ ]:
def audit_numeric_array(values, chunk_rows=65536):
    assert values.ndim == 2 and len(values) > 0 and values.shape[1] > 0
    assert np.issubdtype(values.dtype, np.number)
    nan_by_feature = np.zeros(values.shape[1], dtype=np.int64)
    inf_by_feature = np.zeros(values.shape[1], dtype=np.int64)
    rows_with_nan = 0
    for offset in range(0, len(values), chunk_rows):
        block = np.asarray(values[offset:offset + chunk_rows])
        nan_mask = np.isnan(block)
        nan_by_feature += nan_mask.sum(axis=0)
        inf_by_feature += np.isinf(block).sum(axis=0)
        rows_with_nan += int(nan_mask.any(axis=1).sum())
    return nan_by_feature, inf_by_feature, rows_with_nan

all_stats_rows = []
missing_feature_rows = []

def audit_e1_dataset(dataset_name, train_values, test_values, labels):
    labels = np.asarray(labels)
    assert labels.ndim == 1 or (labels.ndim == 2 and labels.shape[1] == 1)
    labels = labels.reshape(-1)
    assert labels.shape == (len(test_values),)
    assert np.isfinite(labels).all() and np.isin(labels, [0, 1]).all()
    assert train_values.shape[1] == test_values.shape[1]

    for split_name, values in [("train", train_values), ("test", test_values)]:
        nan_counts, inf_counts, rows_with_nan = audit_numeric_array(values)
        is_test = split_name == "test"
        all_stats_rows.append({
            "dataset": dataset_name, "split": split_name,
            "n_points": int(len(values)), "n_features": int(values.shape[1]),
            "dtype": str(values.dtype), "nan_values": int(nan_counts.sum()),
            "features_with_nan": int((nan_counts > 0).sum()), "rows_with_nan": rows_with_nan,
            "inf_values": int(inf_counts.sum()),
            "anomaly_points": int((labels == 1).sum()) if is_test else None,
            "anomaly_rate_point": float(labels.mean()) if is_test else None,
            "anomaly_rate_pct": float(labels.mean() * 100) if is_test else None,
            "label_status": "provided" if is_test else "normal_by_benchmark_protocol_not_measured",
        })
        for feature_index in range(values.shape[1]):
            missing_feature_rows.append({
                "dataset": dataset_name, "split": split_name, "feature_index": feature_index,
                "nan_count": int(nan_counts[feature_index]), "inf_count": int(inf_counts[feature_index]),
            })

# PSM arrays remain the raw arrays from E1, not the normalized train_z/test_z.
audit_e1_dataset("PSM", X_train, X_test, y_test)
for dataset_name in ["MSL", "SMD"]:
    folder = EXTRA_DATA_CACHE / dataset_name
    train_array = np.load(folder / f"{dataset_name}_train.npy", mmap_mode="r", allow_pickle=False)
    test_array = np.load(folder / f"{dataset_name}_test.npy", mmap_mode="r", allow_pickle=False)
    label_array = np.load(folder / f"{dataset_name}_test_label.npy", mmap_mode="r", allow_pickle=False)
    audit_e1_dataset(dataset_name, train_array, test_array, label_array)

e1_all_stats = pd.DataFrame(all_stats_rows)
e1_missing_by_feature = pd.DataFrame(missing_feature_rows)
e1_all_stats.to_csv(E1_EXTRA_RUN / "dataset_stats_all.csv", index=False)
e1_missing_by_feature.to_csv(E1_EXTRA_RUN / "missing_values_by_feature_all.csv", index=False)
for dataset_name in ["PSM", "MSL", "SMD"]:
    e1_all_stats[e1_all_stats["dataset"] == dataset_name].to_csv(
        E1_EXTRA_RUN / f"dataset_stats_{dataset_name}.csv", index=False
    )

notes = """E1: dữ liệu gốc, chưa điền NaN/Inf hay chuẩn hóa MSL/SMD.
Train không có nhãn kèm để đo tỉ lệ bất thường; các ô trống không có nghĩa là 0%.
anomaly_rate_point ở thang 0–1; anomaly_rate_pct ở thang 0–100.
MSL/SMD ở nguồn này là các mảng gộp. Không có metadata ranh giới thực thể trong các file npy đang dùng.
Chưa tạo window hay train MSL/SMD; cần chốt cách xử lý ranh giới chuỗi với nhóm trước khi chạy.
Không suy luận tên/ngữ nghĩa feature từ chỉ số cột.
"""
(E1_EXTRA_RUN / "E1_notes.md").write_text(notes, encoding="utf-8")
display(e1_all_stats)
print("PASS: E1 PSM/MSL/SMD; số chiều và nhãn đã được kiểm tra.")
print("Kết quả E1:", E1_EXTRA_RUN)


,dataset,split,n_points,n_features,dtype,nan_values,features_with_nan,rows_with_nan,inf_values,anomaly_points,anomaly_rate_point,anomaly_rate_pct,label_status
0,PSM,train,132481,25,float64,4195,12,2697,0,NaN,NaN,NaN,normal_by_benchmark_protocol_not_measured
1,PSM,test,87841,25,float64,0,0,0,0,24381.0,0.277558,27.755832,provided
2,MSL,train,58317,55,float64,0,0,0,0,NaN,NaN,NaN,normal_by_benchmark_protocol_not_measured
3,MSL,test,73729,55,float64,0,0,0,0,7766.0,0.105332,10.533169,provided
4,SMD,train,708405,38,float32,0,0,0,0,NaN,NaN,NaN,normal_by_benchmark_protocol_not_measured
5,SMD,test,708420,38,float32,0,0,0,0,29444.0,0.041563,4.156291,provided


PASS: E1 PSM/MSL/SMD; số chiều và nhãn đã được kiểm tra.
Kết quả E1: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E1_all_public_20261122T172712_505585Z


## 13. Gói bàn giao ngày đầu
Chạy sau khi debug và E1 đều đạt. Tạo thư mục mới trên Drive, chỉ chứa kết quả/code/config/log,
không sao chép raw data lớn. **Notebook hiện tại không tự được xuất vào thư mục**:
bấm Ctrl+S và gửi link notebook cùng link thư mục cho nhóm.
Cấu hình preprocessing vẫn là V1 tạm, cần nhóm xác nhận trước chạy hàng loạt.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
DAY1_BUNDLE = DAY1_ROOT / ("DAY1_PSM_MTGFlow_E1_" + stamp)
DAY1_BUNDLE.mkdir(parents=True, exist_ok=False)
assert summary["status"] == "passed"
assert len(e1_all_stats) == 6

shutil.copytree(DEBUG_RUN, DAY1_BUNDLE / "mtgflow_debug")
shutil.copytree(E1_EXTRA_RUN, DAY1_BUNDLE / "E1")
for filename in ["data_manifest.json", "test_windows_w60_s10.csv", "metric_definition.json", "environment.txt"]:
    source = Path(RUN) / filename
    assert source.is_file(), f"Thiếu file bàn giao: {source}"
    shutil.copy2(source, DAY1_BUNDLE / filename)

note = f"""# Bàn giao ngày đầu — Hoàng Khang

## Đã kiểm tra bằng notebook
- E1 PSM/MSL/SMD, checksum, dữ liệu thiếu và nhãn.
- Window test PSM và unit test metric (dữ liệu giả).
- Pipeline PSM theo DAMR public code và DataLoader.
- MTGFlow import được, forward một batch, train đúng {len(training_rows)} batch debug.
- Loss/gradient/trọng số hữu hạn và có cập nhật tham số.
- Checkpoint/scaler tải lại được, score batch kiểm thử khớp.

## Không được hiểu nhầm
- CHƯA train đầy đủ model, CHƯA có AUROC/AUPRC chính thức và CHƯA chạy đủ 3 seed.
- MSL/SMD mới làm E1, CHƯA train hay tạo window cho hai dataset này.
- debug_probe_scores.csv chỉ là score trên {len(probe_frame)} cửa sổ, không phải toàn bộ test.
- Checkpoint debug chưa dùng làm checkpoint thí nghiệm E5 hay kết quả E3a.

## Protocol tái hiện đã triển khai và cần áp dụng thống nhất
- PSM: np.nan_to_num; StandardScaler fit toàn bộ file train trước split window.
- Validation: block window liên tiếp bắt đầu ngẫu nhiên; lưu ID trong window_split.npz.
- Window 60/stride 10; nhãn any-anomaly; AUPRC đang định nghĩa bằng average precision.
- Seed/batch size/epoch và checkpoint policy chính thức chưa được chốt.
- MSL/SMD dạng gộp: kiểm tra ranh giới thực thể trước khi tạo window.

## Code và tái hiện
- Code model gốc: zqhang/MTGFLOW@{MTG_COMMIT}.
- Patch chỉ loại import không dùng cgitb/turtle và chuyển NF sang private package, không sửa kiến trúc/loss.
- Code nguồn đã dùng và diff nằm trong mtgflow_debug/source và compatibility.patch.
- Mở notebook 00_start_psm.ipynb và chạy theo thứ tự. Notebook không tự nằm trong gói này:
  hãy lưu Ctrl+S và gửi link notebook kèm link thư mục.
"""
(DAY1_BUNDLE / "README_DAY1.md").write_text(note, encoding="utf-8")
print("Đã tạo gói bàn giao, không chứa raw data lớn:", DAY1_BUNDLE)
print("Lưu notebook Ctrl+S, rồi gửi nhóm link notebook và link thư mục trên.")

Đã tạo gói bàn giao, không chứa raw data lớn: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/DAY1_PSM_MTGFlow_E1_20261122T172721_370492Z
Lưu notebook Ctrl+S, rồi gửi nhóm link notebook và link thư mục trên.


## 14. MTGFlow — chạy đầy đủ PSM, seed 16

##  Ô A

In [ ]:
import json
import random
import shutil
import time
import joblib
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datetime import datetime, timezone
from torch.utils.data import DataLoader

required = [
    "MTGFLOW_CLASS", "MODEL_CONFIG",
    "train_dataset", "val_dataset", "test_dataset",
    "SAVE", "SEED", "PROTOCOL_ID", "index_df"
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Chạy lại phần chuẩn bị trước. Thiếu: {missing}"

assert torch.cuda.is_available(), (
    "Chưa có GPU CUDA. Dừng tại đây và kiểm tra runtime trước."
)
assert SEED in (16, 17, 18)

# Chặn trường hợp đổi tên seed nhưng vẫn dùng config/split cũ
saved_config = json.loads(
    (Path(SAVE) / "config.json").read_text(encoding="utf-8")
)
assert saved_config["seed"] == SEED, "Chạy lại mục 7–8 cho đúng seed."

with np.load(
    Path(SAVE) / "window_split.npz",
    allow_pickle=False
) as saved_split:
    np.testing.assert_array_equal(
        saved_split["train_ids"],
        np.asarray(train_dataset.indices)
    )
    np.testing.assert_array_equal(
        saved_split["val_ids"],
        np.asarray(val_dataset.indices)
    )
assert PROTOCOL_ID == "DAMR_public_code_PSM_v1"

assert len(train_dataset) == 10595
assert len(val_dataset) == 2648
assert len(test_dataset) == 8779

EPOCHS = 40
FULL_BATCH = 512
LR = 0.002
WEIGHT_DECAY = 0.0005
device = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Tạo model MỚI, không tiếp tục model đã train 3 batch debug
full_model = MTGFLOW_CLASS(**MODEL_CONFIG).to(device)

full_train_loader = DataLoader(
    train_dataset,
    batch_size=FULL_BATCH,
    shuffle=True,
    drop_last=True,
    num_workers=0
)
full_val_loader = DataLoader(
    val_dataset,
    batch_size=FULL_BATCH,
    shuffle=False,
    drop_last=False,
    num_workers=0
)
full_test_loader = DataLoader(
    test_dataset,
    batch_size=FULL_BATCH,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

optimizer_full = torch.optim.Adam(
    full_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
FULL_RUN = Path(
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
) / f"MTGFlow_PSM_seed{SEED}_{stamp}"
FULL_RUN.mkdir(parents=True, exist_ok=False)

for filename in ["config.json", "scaler.joblib", "window_split.npz"]:
    shutil.copy2(Path(SAVE) / filename, FULL_RUN / filename)

run_config = {
    "model": "MTGFlow",
    "dataset": "PSM",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": FULL_BATCH,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "model_config": MODEL_CONFIG,
    "protocol_id": PROTOCOL_ID,
    "checkpoint_policy": "last_epoch_fixed_in_advance",
    "auprc_definition": "average_precision",
    "threshold_policy": "not_selected",
    "evaluation_mode": "model.eval",
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0)
}
(FULL_RUN / "run_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)

print("GPU:", torch.cuda.get_device_name(0))
print("Số batch train mỗi epoch:", len(full_train_loader))
print("Epoch:", EPOCHS)
print("Lưu tại:", FULL_RUN)

GPU: Tesla T4
Số batch train mỗi epoch: 20
Epoch: 40
Lưu tại: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/MTGFlow_PSM_seed18_20261122T172725_410993Z


## Ô B

In [ ]:
epoch_history = []
full_started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    full_model.train()

    total_loss = 0.0
    total_windows = 0

    for batch_number, (batch_x, _, _) in enumerate(
        full_train_loader, start=1
    ):
        batch_x = batch_x.to(device)
        optimizer_full.zero_grad(set_to_none=True)

        loss = -full_model(batch_x)
        assert loss.ndim == 0
        assert torch.isfinite(loss), (
            f"Loss NaN/Inf: epoch {epoch}, batch {batch_number}"
        )

        loss.backward()

        finite_gradients = [
            torch.isfinite(p.grad).all()
            for p in full_model.parameters()
            if p.grad is not None
        ]
        assert finite_gradients
        assert torch.stack(finite_gradients).all().item(), (
            f"Gradient NaN/Inf ở epoch {epoch}"
        )

        torch.nn.utils.clip_grad_value_(full_model.parameters(), 1.0)
        optimizer_full.step()

        total_loss += loss.item() * len(batch_x)
        total_windows += len(batch_x)

    epoch_seconds = time.perf_counter() - epoch_started

    epoch_history.append({
        "epoch": epoch,
        "train_loss": total_loss / total_windows,
        "trained_windows": total_windows,
        "epoch_seconds": epoch_seconds
    })

    # Ghi checkpoint qua file tạm rồi thay bản last của chính lượt này
    checkpoint_tmp = FULL_RUN / "checkpoint_last.tmp"
    torch.save({
        "epoch": epoch,
        "model_state_dict": full_model.state_dict(),
        "optimizer_state_dict": optimizer_full.state_dict(),
        "model_config": MODEL_CONFIG,
        "run_config": run_config
    }, checkpoint_tmp)
    checkpoint_tmp.replace(FULL_RUN / "checkpoint_last.pt")

    pd.DataFrame(epoch_history).to_csv(
        FULL_RUN / "epoch_history.csv", index=False
    )

    message = (
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={total_loss / total_windows:.6f} | "
        f"{epoch_seconds:.1f}s"
    )
    print(message, flush=True)

    with (FULL_RUN / "train.log").open("a", encoding="utf-8") as f:
        f.write(message + "\n")

train_runtime_s = time.perf_counter() - full_started

print("TRAIN HOÀN THÀNH:", EPOCHS, "epoch")
print("Thời gian train:", round(train_runtime_s / 60, 2), "phút")
print("Checkpoint:", FULL_RUN / "checkpoint_last.pt")

Epoch 01/40 | loss=1.157897 | 2.0s
Epoch 02/40 | loss=-0.867011 | 2.0s
Epoch 03/40 | loss=-1.911173 | 2.1s
Epoch 04/40 | loss=-2.431491 | 2.1s
Epoch 05/40 | loss=-2.849377 | 2.0s
Epoch 06/40 | loss=-3.052170 | 2.1s
Epoch 07/40 | loss=-3.144627 | 2.1s
Epoch 08/40 | loss=-3.207614 | 2.1s
Epoch 09/40 | loss=-3.242952 | 2.1s
Epoch 10/40 | loss=-3.272901 | 2.1s
Epoch 11/40 | loss=-3.292823 | 2.0s
Epoch 12/40 | loss=-3.317807 | 2.0s
Epoch 13/40 | loss=-3.333670 | 2.0s
Epoch 14/40 | loss=-3.351672 | 2.1s
Epoch 15/40 | loss=-3.356367 | 2.2s
Epoch 16/40 | loss=-3.370698 | 2.0s
Epoch 17/40 | loss=-3.383354 | 2.0s
Epoch 18/40 | loss=-3.391758 | 2.0s
Epoch 19/40 | loss=-3.405797 | 1.9s
Epoch 20/40 | loss=-3.413116 | 1.9s
Epoch 21/40 | loss=-3.417304 | 1.9s
Epoch 22/40 | loss=-3.421707 | 1.9s
Epoch 23/40 | loss=-3.433208 | 2.0s
Epoch 24/40 | loss=-3.435572 | 2.0s
Epoch 25/40 | loss=-3.447649 | 1.9s
Epoch 26/40 | loss=-3.451827 | 1.9s
Epoch 27/40 | loss=-3.445430 | 1.9s
Epoch 28/40 | loss=-3.439233 

## Ô C

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

checkpoint = torch.load(
    FULL_RUN / "checkpoint_last.pt",
    map_location=device,
    weights_only=True
)
assert checkpoint["epoch"] == EPOCHS

eval_model = MTGFLOW_CLASS(**checkpoint["model_config"]).to(device)
eval_model.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
eval_model.eval()


def collect_window_scores(model, loader, split_name):
    # Validation là Subset; test là Dataset trực tiếp
    dataset = loader.dataset
    if isinstance(dataset, torch.utils.data.Subset):
        base = dataset.dataset
        expected_ids = np.asarray(dataset.indices)
    else:
        base = dataset
        expected_ids = np.arange(len(dataset))

    frames = []

    with torch.no_grad():
        for batch_x, labels, window_ids in loader:
            scores = -model.test(batch_x.to(device))

            assert scores.ndim == 1
            assert len(scores) == len(batch_x)
            assert torch.isfinite(scores).all()

            ids = window_ids.numpy()
            frames.append(pd.DataFrame({
                "split": split_name,
                "window_id": ids,
                "start": base.starts[ids],
                "end_exclusive": base.starts[ids] + base.window,
                "label": labels.numpy(),
                "score": scores.cpu().numpy()
            }))

    result = pd.concat(frames, ignore_index=True)
    assert not result["window_id"].duplicated().any()
    result = result.sort_values("window_id").reset_index(drop=True)

    np.testing.assert_array_equal(
        result["window_id"].to_numpy(),
        np.sort(expected_ids)
    )
    return result


evaluation_started = time.perf_counter()

validation_scores = collect_window_scores(
    eval_model, full_val_loader, "validation"
)
test_scores = collect_window_scores(
    eval_model, full_test_loader, "test"
)

# Đối chiếu cả ID và đáp án với bảng test đã tạo từ E1
truth = index_df.sort_values("window_id").reset_index(drop=True)

np.testing.assert_array_equal(
    test_scores["window_id"].to_numpy(),
    truth["window_id"].to_numpy()
)
np.testing.assert_array_equal(
    test_scores["label"].to_numpy(),
    truth["label"].to_numpy()
)

validation_scores.to_csv(FULL_RUN / "validation_scores.csv", index=False)
test_scores.to_csv(FULL_RUN / "test_scores.csv", index=False)

y = test_scores["label"].to_numpy()
scores = test_scores["score"].to_numpy()
assert len(np.unique(y)) == 2

evaluation_runtime_s = time.perf_counter() - evaluation_started

metrics_row = {
    "model": "MTGFlow",
    "variant": "base",
    "dataset": "PSM",
    "seed": SEED,
    "noise": 0,
    "epoch": EPOCHS,
    "auroc": float(roc_auc_score(y, scores)),
    "auprc": float(average_precision_score(y, scores)),
    "f1": np.nan,
    "flagged_rate": np.nan,
    "runtime_s": train_runtime_s + evaluation_runtime_s
}

metrics_table = pd.DataFrame([metrics_row])
metrics_table.to_csv(FULL_RUN / "metrics.csv", index=False)

(FULL_RUN / "completed.json").write_text(
    json.dumps({
        "status": "completed",
        "checkpoint_policy": "last_epoch",
        "validation_windows": len(validation_scores),
        "test_windows": len(test_scores),
        "train_runtime_s": train_runtime_s,
        "evaluation_runtime_s": evaluation_runtime_s,
        "f1_status": "not_computed_threshold_not_selected"
    }, indent=2),
    encoding="utf-8"
)

display(metrics_table)
print("Validation windows:", len(validation_scores))
print("Test windows:", len(test_scores))
print("HOÀN THÀNH MỘT LƯỢT E3a:", FULL_RUN)

,model,variant,dataset,seed,noise,epoch,auroc,auprc,f1,flagged_rate,runtime_s
0,MTGFlow,base,PSM,18,0,40,0.739351,0.586612,NaN,NaN,82.139952


Validation windows: 2648
Test windows: 8779
HOÀN THÀNH MỘT LƯỢT E3a: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/MTGFlow_PSM_seed18_20261122T172725_410993Z


## 15. Tổng hợp và bàn giao MTGFlow–PSM, ba seed

In [ ]:
import json
import shutil
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from sklearn.metrics import roc_auc_score, average_precision_score

ROOT = Path(
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
)
SEEDS = [16, 17, 18]

selected_runs = {}
metric_rows = []
reference_settings = None

# Các thiết lập phải giống nhau giữa ba seed
setting_keys = [
    "model", "dataset", "epochs", "batch_size",
    "lr", "weight_decay", "model_config", "protocol_id",
    "checkpoint_policy", "auprc_definition",
    "threshold_policy", "evaluation_mode"
]

required_files = [
    "completed.json", "run_config.json", "config.json",
    "metrics.csv", "epoch_history.csv", "train.log",
    "checkpoint_last.pt", "scaler.joblib", "window_split.npz",
    "validation_scores.csv", "test_scores.csv"
]

for seed in SEEDS:
    candidates = sorted(
        ROOT.glob(f"MTGFlow_PSM_seed{seed}_*"),
        reverse=True
    )

    chosen = None

    # Chọn lượt hoàn tất mới nhất, KHÔNG chọn theo AUROC cao nhất
    for folder in candidates:
        if not all((folder / name).is_file() for name in required_files):
            continue

        cfg = json.loads(
            (folder / "run_config.json").read_text(encoding="utf-8")
        )
        status = json.loads(
            (folder / "completed.json").read_text(encoding="utf-8")
        )

        if (
            status.get("status") == "completed"
            and cfg.get("seed") == seed
            and cfg.get("model") == "MTGFlow"
            and cfg.get("dataset") == "PSM"
            and cfg.get("epochs") == 40
            and cfg.get("protocol_id") == "DAMR_public_code_PSM_v1"
            and cfg.get("checkpoint_policy") == "last_epoch_fixed_in_advance"
        ):
            chosen = folder
            break

    assert chosen is not None, f"Chưa có lượt hoàn tất hợp lệ cho seed {seed}"

    settings = {key: cfg[key] for key in setting_keys}
    if reference_settings is None:
        reference_settings = settings
    else:
        assert settings == reference_settings, (
            f"Cấu hình seed {seed} khác các seed trước. Kiểm tra trước khi gom."
        )

    preprocessing = json.loads(
        (chosen / "config.json").read_text(encoding="utf-8")
    )
    assert preprocessing["seed"] == seed
    assert preprocessing["scaler_fit_on"] == (
        "full_train_file_before_window_split"
    )

    row = pd.read_csv(chosen / "metrics.csv")
    assert len(row) == 1
    assert int(row.iloc[0]["seed"]) == seed
    assert int(row.iloc[0]["epoch"]) == 40

    test = pd.read_csv(chosen / "test_scores.csv")
    validation = pd.read_csv(chosen / "validation_scores.csv")

    assert len(test) == 8779
    assert len(validation) == 2648
    assert not test["window_id"].duplicated().any()
    assert not validation["window_id"].duplicated().any()
    assert np.isfinite(test["score"]).all()
    assert np.isfinite(validation["score"]).all()
    assert test["label"].isin([0, 1]).all()

    np.testing.assert_array_equal(
        np.sort(test["window_id"].to_numpy()),
        np.arange(8779)
    )

    with np.load(chosen / "window_split.npz", allow_pickle=False) as split:
        np.testing.assert_array_equal(
            np.sort(validation["window_id"].to_numpy()),
            np.sort(split["val_ids"])
        )

    # Kiểm tra metric đã lưu khớp với chính file score
    np.testing.assert_allclose(
        row.iloc[0]["auroc"],
        roc_auc_score(test["label"], test["score"]),
        atol=1e-10
    )
    np.testing.assert_allclose(
        row.iloc[0]["auprc"],
        average_precision_score(test["label"], test["score"]),
        atol=1e-10
    )

    selected_runs[seed] = chosen
    metric_rows.append(row)
    print(f"Seed {seed}: {chosen.name}")

results = pd.concat(metric_rows, ignore_index=True)
results = results.sort_values("seed").reset_index(drop=True)

# Tạo gói mới, không thay/xóa các lượt gốc
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
PACKAGE = ROOT / f"BAN_GIAO_MTGFlow_PSM_3seeds_{stamp}"
PACKAGE.mkdir(parents=True, exist_ok=False)

for seed, folder in selected_runs.items():
    shutil.copytree(folder, PACKAGE / f"seed_{seed}")

# Giữ đúng các cột kết quả theo kế hoạch nhóm
results.to_csv(PACKAGE / "results.csv", index=False)

summary = {
    "model": "MTGFlow",
    "dataset": "PSM",
    "n_seeds": len(results),
    "seeds": "16,17,18",
    "epoch": 40,
    "checkpoint_policy": "last_epoch"
}

for metric in ["auroc", "auprc", "f1", "flagged_rate"]:
    summary[f"{metric}_mean"] = results[metric].mean()
    summary[f"{metric}_std"] = results[metric].std(ddof=1)

pd.DataFrame([summary]).to_csv(
    PACKAGE / "summary.csv", index=False
)

pd.DataFrame([
    {"seed": seed, "original_run": str(folder)}
    for seed, folder in selected_runs.items()
]).to_csv(PACKAGE / "selected_runs.csv", index=False)

readme = """# Bàn giao MTGFlow trên PSM — 3 seed

- Seed: 16, 17, 18; 40 epoch; dùng checkpoint epoch cuối.
- results.csv: kết quả riêng từng seed.
- summary.csv: mean và sample std (ddof=1).
- Mỗi thư mục seed chứa checkpoint, scaler, split, config, log,
  score validation và toàn bộ test.
- AUPRC được tính bằng average precision.
- F1/flagged_rate để trống nếu chưa chọn ngưỡng, không phải bằng 0.
- Nhãn validation -1 là ký hiệu không có nhãn gốc.
- Nếu có nhiều lượt hợp lệ cùng seed, dùng lượt hoàn tất mới nhất,
  không chọn theo metric cao nhất.
- Đây mới là MTGFlow–PSM: 3/45 lượt E3a, chưa hoàn thành tất cả baseline.
- Gửi kèm link notebook để có code model/adapter và hướng dẫn chạy.
"""
(PACKAGE / "README.md").write_text(readme, encoding="utf-8")

display(results)
display(pd.DataFrame([summary]))
print("GÓI BÀN GIAO:", PACKAGE)

Seed 16: MTGFlow_PSM_seed16_20261122T164705_499086Z
Seed 17: MTGFlow_PSM_seed17_20261122T172442_322830Z
Seed 18: MTGFlow_PSM_seed18_20261122T172725_410993Z


,model,variant,dataset,seed,noise,epoch,auroc,auprc,f1,flagged_rate,runtime_s
0,MTGFlow,base,PSM,16,0,40,0.697329,0.546988,NaN,NaN,81.924330
1,MTGFlow,base,PSM,17,0,40,0.695151,0.516385,NaN,NaN,82.664461
2,MTGFlow,base,PSM,18,0,40,0.739351,0.586612,NaN,NaN,82.139952


,model,dataset,n_seeds,seeds,epoch,checkpoint_policy,auroc_mean,auroc_std,auprc_mean,auprc_std,f1_mean,f1_std,flagged_rate_mean,flagged_rate_std
0,MTGFlow,PSM,3,"16,17,18",40,last_epoch,0.710611,0.024914,0.549995,0.03521,NaN,NaN,NaN,NaN


GÓI BÀN GIAO: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/BAN_GIAO_MTGFlow_PSM_3seeds_20261122T172945_894278Z


## 16. MTGFlow trên MSL và SMD — seed 16, 17, 18

In [ ]:
import gc
import hashlib
import json
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset


def run_mtgflow_extra(dataset_name, seed):
    assert dataset_name in ("MSL", "SMD")
    assert seed in (16, 17, 18)

    needed = [
        "MTGFLOW_CLASS", "PSMWindowDataset", "EXTRA_DATA_REFERENCES",
        "MODEL_LOCAL", "MODEL_CACHE", "MTG_COMMIT", "DEBUG_RUN"
    ]
    missing = [k for k in needed if k not in globals()]
    assert not missing, (
        f"Thiếu {missing}: chạy phần import model và tải MSL/SMD trước."
    )
    assert torch.cuda.is_available(), "Chưa có CUDA; không bắt đầu train."

    device = torch.device("cuda:0")
    window, stride, val_ratio = 60, 10, 0.2
    epochs, batch_size = 40, 512
    lr, weight_decay = 0.002, 0.0005

    data_root = Path("/content/drive/MyDrive/CS313_Nhom5/data")
    root = Path(
        "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
    )
    assert data_root.is_dir()
    assert Path(MODEL_LOCAL).is_dir()

    for name in ["compatibility.patch", "source_provenance.json"]:
        assert (Path(DEBUG_RUN) / name).is_file(), (
            f"Thiếu nguồn tham chiếu: {name}"
        )

    # 1. Đọc dữ liệu đã có trên Drive và kiểm tra checksum.
    arrays, manifest = {}, {}

    for split in ["train", "test", "test_label"]:
        key = f"{dataset_name}/{dataset_name}_{split}.npy"
        path = data_root / key
        assert path.is_file(), f"Thiếu dữ liệu: {path}"

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)

        assert digest.hexdigest() == (
            EXTRA_DATA_REFERENCES[key]["sha256"]
        ), f"Checksum không khớp: {path}"

        manifest[key] = {
            **EXTRA_DATA_REFERENCES[key], "path": str(path)
        }
        arrays[split] = np.load(path, allow_pickle=False)

    raw_train, raw_test = arrays["train"], arrays["test"]
    raw_labels = arrays["test_label"]

    assert raw_labels.ndim == 1 or (
        raw_labels.ndim == 2 and raw_labels.shape[1] == 1
    )
    labels = raw_labels.reshape(-1)

    expected_shape = {
        "MSL": (58317, 73729, 55),
        "SMD": (708405, 708420, 38)
    }
    n_train, n_test, features = expected_shape[dataset_name]

    assert raw_train.shape == (n_train, features)
    assert raw_test.shape == (n_test, features)
    assert labels.shape == (n_test,)
    assert np.isin(labels, [0, 1]).all()
    assert np.isfinite(raw_train).all()
    assert np.isfinite(raw_test).all()

    # 2. DAMR MSL/SMD giữ mọi feature, không bỏ cột đầu như PSM.
    # Loader gốc không có bước điền NaN cho hai dataset này.
    scaler = StandardScaler().fit(raw_train)
    train_z = scaler.transform(raw_train).astype(np.float32)
    test_z = scaler.transform(raw_test).astype(np.float32)

    assert np.isfinite(train_z).all()
    assert np.isfinite(test_z).all()

    # Tên lớp có chữ PSM nhưng cách cắt window dùng được cho mọi số feature.
    base = PSMWindowDataset(
        train_z, window=window, stride=stride
    )
    test_ds = PSMWindowDataset(
        test_z, labels=labels, window=window, stride=stride
    )

    # 3. Chia validation theo đúng block window ngẫu nhiên của DAMR.
    n = len(base)
    val_size = int(n * val_ratio)
    val_start = random.Random(seed).randrange(
        int(n * (1 - val_ratio))
    )

    ids = np.arange(n, dtype=np.int64)
    val_ids = ids[val_start:val_start + val_size]
    train_ids = np.concatenate([
        ids[:val_start], ids[val_start + val_size:]
    ])

    assert np.intersect1d(train_ids, val_ids).size == 0
    np.testing.assert_array_equal(
        np.sort(np.r_[train_ids, val_ids]), ids
    )

    train_ds = Subset(base, train_ids.tolist())
    val_ds = Subset(base, val_ids.tolist())

    # 4. Tạo model/optimizer MỚI cho từng dataset–seed.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model_config = {
        "n_blocks": 1,
        "input_size": 1,
        "hidden_size": 32,
        "n_hidden": 1,
        "window_size": window,
        "n_sensor": features,
        "dropout": 0.0,
        "model": "MAF",
        "batch_norm": False
    }

    model = MTGFLOW_CLASS(**model_config).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=True, drop_last=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size,
        shuffle=False, num_workers=0
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size,
        shuffle=False, num_workers=0
    )
    assert len(train_loader) > 0

    # 5. Mỗi lượt có thư mục mới, không ghi đè các lượt trước.
    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
    out = root / f"MTGFlow_{dataset_name}_seed{seed}_{stamp}"
    out.mkdir(parents=True, exist_ok=False)

    cfg = {
        "model": "MTGFlow",
        "dataset": dataset_name,
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "weight_decay": weight_decay,
        "model_config": model_config,
        "protocol_id": f"DAMR_public_code_{dataset_name}_v1",
        "checkpoint_policy": "last_epoch_fixed_in_advance",
        "auprc_definition": "average_precision",
        "threshold_policy": "not_selected",
        "evaluation_mode": "model.eval",
        "torch": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(0),
        "model_commit": MTG_COMMIT
    }

    prep = {
        "dataset": dataset_name,
        "seed": seed,
        "protocol_id": cfg["protocol_id"],
        "window": window,
        "stride": stride,
        "val_ratio": val_ratio,
        "scaler_fit_on": "full_train_file_before_window_split",
        "missing_policy": "no_imputation_assert_finite_as_DAMR_MSL_SMD",
        "split_policy": "random_start_contiguous_validation_window_block",
        "train_file_rows": n_train,
        "test_rows": n_test,
        "n_features": features,
        "full_train_windows": n,
        "train_windows": len(train_ds),
        "validation_windows": len(val_ds),
        "test_windows": len(test_ds),
        "val_start_window_id": val_start,
        "sequence_boundary_policy": "merged_arrays_no_boundary_metadata"
    }

    for name, value in [
        ("run_config.json", cfg),
        ("config.json", prep),
        ("data_manifest.json", manifest)
    ]:
        (out / name).write_text(
            json.dumps(value, indent=2), encoding="utf-8"
        )

    joblib.dump(scaler, out / "scaler.joblib")
    restored_scaler = joblib.load(out / "scaler.joblib")
    assert int(restored_scaler.n_samples_seen_) == n_train

    for raw, scaled in [(raw_train, train_z), (raw_test, test_z)]:
        np.testing.assert_allclose(
            restored_scaler.transform(raw).astype(np.float32),
            scaled
        )

    np.savez(
        out / "window_split.npz",
        train_ids=train_ids,
        val_ids=val_ids,
        val_start_index=val_start
    )

    shutil.copytree(MODEL_LOCAL, out / "source")
    shutil.copy2(
        Path(MODEL_CACHE) / "LICENSE",
        out / "source" / "LICENSE"
    )
    for name in ["compatibility.patch", "source_provenance.json"]:
        shutil.copy2(Path(DEBUG_RUN) / name, out / name)

    (out / "README.md").write_text(
        "MTGFlow; last epoch; AP=average precision; F1 chưa chọn ngưỡng.\n"
        "Dữ liệu là mảng gộp, không có ranh giới chuỗi/máy; "
        "window có thể cắt qua ranh giới.\n"
        "Train/val có thể chồng lấn điểm gốc, giống split window DAMR.\n"
        "Gửi kèm notebook chứa runner; source/ chỉ chứa code model.\n",
        encoding="utf-8"
    )

    print(
        f"{dataset_name}, seed={seed}: "
        f"train/val/test={len(train_ds)}/{len(val_ds)}/{len(test_ds)}"
    )
    print("Thư mục:", out, flush=True)

    # 6. Train đủ 40 epoch, không chọn epoch theo test.
    history = []
    torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, count = 0.0, 0

        for x, _, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)

            loss = -model(x)
            assert loss.ndim == 0 and torch.isfinite(loss).item()

            loss.backward()
            grads = [
                p.grad for p in model.parameters()
                if p.grad is not None
            ]
            assert grads and all(
                torch.isfinite(g).all().item() for g in grads
            )

            torch.nn.utils.clip_grad_value_(
                model.parameters(), 1.0
            )
            optimizer.step()

            total_loss += loss.item() * len(x)
            count += len(x)

        history.append({
            "epoch": epoch,
            "train_loss": total_loss / count,
            "trained_windows": count
        })

        temporary = out / "checkpoint_last.tmp"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "model_config": model_config,
            "run_config": cfg
        }, temporary)
        temporary.replace(out / "checkpoint_last.pt")

        pd.DataFrame(history).to_csv(
            out / "epoch_history.csv", index=False
        )

        message = (
            f"{dataset_name} seed{seed}: "
            f"epoch {epoch:02d}/{epochs}, "
            f"loss={total_loss / count:.6f}"
        )
        print(message, flush=True)
        with (out / "train.log").open("a", encoding="utf-8") as f:
            f.write(message + "\n")

    torch.cuda.synchronize()
    train_seconds = time.perf_counter() - started
    eval_started = time.perf_counter()

    # 7. Nạp checkpoint vào model mới và đối chiếu score.
    ckpt = torch.load(
        out / "checkpoint_last.pt",
        map_location=device,
        weights_only=True
    )
    assert ckpt["epoch"] == epochs

    restored = MTGFLOW_CLASS(**ckpt["model_config"]).to(device)
    restored.load_state_dict(
        ckpt["model_state_dict"], strict=True
    )

    model.eval()
    restored.eval()
    probe = next(iter(test_loader))[0][:8].to(device)

    with torch.no_grad():
        torch.testing.assert_close(
            model.test(probe),
            restored.test(probe),
            rtol=1e-5,
            atol=1e-6
        )

    # 8. Chấm toàn bộ validation/test, giữ ID và vị trí window.
    def score(loader, base_ds, expected_ids, split_name):
        frames = []

        with torch.no_grad():
            for x, y, window_ids in loader:
                values = -restored.test(x.to(device))
                assert values.shape == (len(x),)
                assert torch.isfinite(values).all().item()

                idx = window_ids.numpy()
                frames.append(pd.DataFrame({
                    "split": split_name,
                    "window_id": idx,
                    "start": base_ds.starts[idx],
                    "end_exclusive": base_ds.starts[idx] + window,
                    "label": y.numpy(),
                    "score": values.cpu().numpy()
                }))

        frame = (
            pd.concat(frames, ignore_index=True)
            .sort_values("window_id")
            .reset_index(drop=True)
        )
        np.testing.assert_array_equal(
            frame["window_id"].to_numpy(),
            np.sort(expected_ids)
        )
        assert not frame["window_id"].duplicated().any()

        frame.to_csv(
            out / f"{split_name}_scores.csv", index=False
        )
        return frame

    validation = score(val_loader, base, val_ids, "validation")
    test = score(
        test_loader, test_ds, np.arange(len(test_ds)), "test"
    )

    # Tính nhãn window độc lập để kiểm tra alignment.
    cumulative = np.r_[0, np.cumsum(labels == 1)]
    starts = np.arange(0, len(labels) - window + 1, stride)
    truth = (
        cumulative[starts + window] - cumulative[starts] > 0
    ).astype(int)

    np.testing.assert_array_equal(
        test["start"].to_numpy(), starts
    )
    np.testing.assert_array_equal(
        test["label"].to_numpy(), truth
    )

    torch.cuda.synchronize()
    evaluation_seconds = time.perf_counter() - eval_started

    # 9. Xuất metric đúng schema chung.
    row = {
        "model": "MTGFlow",
        "variant": "base",
        "dataset": dataset_name,
        "seed": seed,
        "noise": 0,
        "epoch": epochs,
        "auroc": float(roc_auc_score(truth, test["score"])),
        "auprc": float(
            average_precision_score(truth, test["score"])
        ),
        "f1": np.nan,
        "flagged_rate": np.nan,
        "runtime_s": train_seconds + evaluation_seconds
    }

    pd.DataFrame([row]).to_csv(
        out / "metrics.csv", index=False
    )
    (out / "completed.json").write_text(
        json.dumps({
            "status": "completed",
            "checkpoint_policy": "last_epoch",
            "validation_windows": len(validation),
            "test_windows": len(test),
            "train_runtime_s": train_seconds,
            "evaluation_runtime_s": evaluation_seconds,
            "f1_status": "not_computed_threshold_not_selected"
        }, indent=2),
        encoding="utf-8"
    )

    print(pd.DataFrame([row]).to_string(index=False))
    print("HOÀN THÀNH:", out)

    del model, restored, optimizer, ckpt, probe, x, loss, grads
    gc.collect()
    torch.cuda.empty_cache()
    return out

In [ ]:
MSL_SEED16_RUN = run_mtgflow_extra("MSL", 16)

MSL, seed=16: train/val/test=4661/1165/7367
Thư mục: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/MTGFlow_MSL_seed16_20261122T174222_258964Z
MSL seed16: epoch 01/40, loss=1.432778
MSL seed16: epoch 02/40, loss=0.682911
MSL seed16: epoch 03/40, loss=-0.088951
MSL seed16: epoch 04/40, loss=-0.489146
MSL seed16: epoch 05/40, loss=-0.923830
MSL seed16: epoch 06/40, loss=-1.452547
MSL seed16: epoch 07/40, loss=-1.496605
MSL seed16: epoch 08/40, loss=-1.924914
MSL seed16: epoch 09/40, loss=-2.383603
MSL seed16: epoch 10/40, loss=-2.567451
MSL seed16: epoch 11/40, loss=0.874689
MSL seed16: epoch 12/40, loss=-0.694759
MSL seed16: epoch 13/40, loss=-1.849722
MSL seed16: epoch 14/40, loss=-1.947061
MSL seed16: epoch 15/40, loss=-2.009727
MSL seed16: epoch 16/40, loss=-2.054277
MSL seed16: epoch 17/40, loss=-2.071372
MSL seed16: epoch 18/40, loss=-2.070186
MSL seed16: epoch 19/40, loss=-2.087064
MSL seed16: epoch 20/40, loss=-2.131315
MSL seed16: epoch 21/40, loss=-2.127961
MSL seed16: epoc

In [ ]:
# Chạy SMD–seed16 trước, sau đó hoàn thành các seed còn thiếu.
# Không chạy lại PSM hoặc MSL–seed16.

remaining_jobs = [
    ("SMD", 16),
    ("MSL", 17),
    ("MSL", 18),
    ("SMD", 17),
    ("SMD", 18),
]

remaining_run_folders = []

for dataset_name, seed in remaining_jobs:
    print(f"\n===== BẮT ĐẦU: {dataset_name}, seed {seed} =====",
          flush=True)

    folder = run_mtgflow_extra(dataset_name, seed)
    remaining_run_folders.append(folder)

print("\nĐÃ HOÀN THÀNH NĂM LƯỢT CÒN LẠI:")
for folder in remaining_run_folders:
    print(folder)


===== BẮT ĐẦU: SMD, seed 16 =====
SMD, seed=16: train/val/test=56668/14167/70837
Thư mục: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/MTGFlow_SMD_seed16_20261122T174502_360523Z
SMD seed16: epoch 01/40, loss=-0.804427
SMD seed16: epoch 02/40, loss=-2.319398
SMD seed16: epoch 03/40, loss=-2.532783
SMD seed16: epoch 04/40, loss=-2.622833
SMD seed16: epoch 05/40, loss=-2.678033
SMD seed16: epoch 06/40, loss=-2.718787
SMD seed16: epoch 07/40, loss=-2.752110
SMD seed16: epoch 08/40, loss=-2.782174
SMD seed16: epoch 09/40, loss=-2.809080
SMD seed16: epoch 10/40, loss=-2.834126
SMD seed16: epoch 11/40, loss=-2.855227
SMD seed16: epoch 12/40, loss=-2.875507
SMD seed16: epoch 13/40, loss=-2.893572
SMD seed16: epoch 14/40, loss=-2.909787
SMD seed16: epoch 15/40, loss=-2.925520
SMD seed16: epoch 16/40, loss=-2.939459
SMD seed16: epoch 17/40, loss=-2.952583
SMD seed16: epoch 18/40, loss=-2.965497
SMD seed16: epoch 19/40, loss=-2.977736
SMD seed16: epoch 20/40, loss=-2.990752
SMD seed16: epo

## 17. DeepSVDD — PSM, MSL, SMD × seed 16, 17, 18

In [ ]:
import ast
import gc
import hashlib
import json
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import DataLoader, Subset

SVDD_COMMIT = "f8cb9ea225fd7e64cfb4556dae3525accac285f5"

assert Path("/content/drive/MyDrive").is_dir(), "Mount Drive trước."

SVDD_CACHE = (
    Path("/content/drive/MyDrive/CS313_Nhom5/source_cache")
    / ("DeepSVDD_" + SVDD_COMMIT)
)
SVDD_CACHE.mkdir(parents=True, exist_ok=True)

SVDD_HASHES = {
    "models/GAN.py":
        "2ba491d5e94e880660ae584d6cac045d6f85dea9a864f4f2fab8a71bc04b5e5c",
    "models/DeepSAD.py":
        "8dcfc3a40edfebb03c8ea3935caa4066358e283047f8696c1bcd08a481aac965",
    "train_other_model.py":
        "db2a383e95521c9ad69b2abf60848e1e90242c4088fc5ec68982783390f1ec11",
    "LICENSE":
        "23a15f9f18973b2376cb281cb3bba6fea79c80c1af79982a48920b40f3a25f4f",
}

for filename, expected in SVDD_HASHES.items():
    path = SVDD_CACHE / filename

    if not path.exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/zqhang/MTGFLOW/"
            f"{SVDD_COMMIT}/{filename}",
            timeout=(30, 120)
        )
        response.raise_for_status()

        assert hashlib.sha256(response.content).hexdigest() == expected

        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)

    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
    print("Source verified:", filename)

# Chỉ nạp ba định nghĩa mạng; không chạy trainer/script gốc.
source = (SVDD_CACHE / "models/GAN.py").read_text(encoding="utf-8")
names = {"ConvEncoder", "ConvDecoder", "CNNAE"}

nodes = [
    node for node in ast.parse(source).body
    if isinstance(node, (ast.FunctionDef, ast.ClassDef))
    and node.name in names
]
assert {node.name for node in nodes} == names

network_code = (
    "import torch\nimport torch.nn as nn\n\n"
    + "\n\n".join(ast.unparse(node) for node in nodes)
    + "\n"
)

SVDD_NETWORK_NS = {}
exec(
    compile(network_code, "svdd_reference_network.py", "exec"),
    SVDD_NETWORK_NS
)
SVDD_CNNAE = SVDD_NETWORK_NS["CNNAE"]

print("Đã nạp encoder CNN tham chiếu. Chưa train.")

Source verified: models/GAN.py
Source verified: models/DeepSAD.py
Source verified: train_other_model.py
Source verified: LICENSE
Đã nạp encoder CNN tham chiếu. Chưa train.


In [ ]:
def run_deepsvdd(dataset_name, seed):
    assert dataset_name in ("PSM", "MSL", "SMD")
    assert seed in (16, 17, 18)
    assert torch.cuda.is_available(), "Chưa có CUDA."
    assert "PSMWindowDataset" in globals(), (
        "Chạy ô định nghĩa PSMWindowDataset trước."
    )

    required_meta = (
        "PSM_EXPECTED_SHA"
        if dataset_name == "PSM"
        else "EXTRA_DATA_REFERENCES"
    )
    assert required_meta in globals(), f"Thiếu {required_meta}"

    device = torch.device("cuda:0")
    root = Path(
        "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
    )
    data_dir = (
        Path("/content/drive/MyDrive/CS313_Nhom5/data")
        / dataset_name
    )

    # 1. Tìm lượt MTGFlow hoàn tất để dùng đúng scaler/split.
    # Chọn lượt hợp lệ mới nhất, không chọn theo metric cao nhất.
    required = [
        "completed.json", "run_config.json", "config.json",
        "metrics.csv", "scaler.joblib", "window_split.npz",
        "validation_scores.csv", "test_scores.csv"
    ]

    reference = None

    for folder in sorted(
        root.glob(f"MTGFlow_{dataset_name}_seed{seed}_*"),
        reverse=True
    ):
        if not all((folder / name).is_file() for name in required):
            continue

        cfg = json.loads((folder / "run_config.json").read_text())
        status = json.loads((folder / "completed.json").read_text())

        if (
            status.get("status") == "completed"
            and cfg.get("seed") == seed
            and cfg.get("dataset") == dataset_name
            and cfg.get("model") == "MTGFlow"
            and cfg.get("epochs") == 40
            and cfg.get("checkpoint_policy")
                == "last_epoch_fixed_in_advance"
            and cfg.get("protocol_id")
                == f"DAMR_public_code_{dataset_name}_v1"
        ):
            reference = folder
            break

    assert reference is not None, (
        f"Không tìm được MTGFlow hoàn tất: {dataset_name}, seed{seed}"
    )

    prep = json.loads((reference / "config.json").read_text())

    assert prep["dataset"] == dataset_name
    assert prep["seed"] == seed
    assert prep["window"] == 60 and prep["stride"] == 10
    assert prep["val_ratio"] == 0.2
    assert prep["scaler_fit_on"] == (
        "full_train_file_before_window_split"
    )

    # 2. Kiểm tra đúng dữ liệu đã ghim.
    if dataset_name == "PSM":
        filenames = ["train.csv", "test.csv", "test_label.csv"]
    else:
        filenames = [
            f"{dataset_name}_{s}.npy"
            for s in ["train", "test", "test_label"]
        ]

    data_manifest = {}

    for filename in filenames:
        path = data_dir / filename
        assert path.is_file(), path

        if dataset_name == "PSM":
            expected = PSM_EXPECTED_SHA[filename]
        else:
            expected = EXTRA_DATA_REFERENCES[
                f"{dataset_name}/{filename}"
            ]["sha256"]

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024), b""
            ):
                digest.update(block)

        assert digest.hexdigest() == expected, path
        data_manifest[filename] = {
            "path": str(path), "sha256": expected
        }

    # 3. Đọc dữ liệu theo protocol đã dùng.
    if dataset_name == "PSM":
        train_df = pd.read_csv(data_dir / "train.csv")
        test_df = pd.read_csv(data_dir / "test.csv")
        label_df = pd.read_csv(data_dir / "test_label.csv")

        assert "timestamp" in str(train_df.columns[0]).lower()
        assert train_df.columns.tolist() == test_df.columns.tolist()
        assert label_df.shape[1] == 2

        np.testing.assert_array_equal(
            test_df.iloc[:, 0], label_df.iloc[:, 0]
        )

        raw_train = np.nan_to_num(train_df.values[:, 1:])
        raw_test = np.nan_to_num(test_df.values[:, 1:])
        labels = label_df.iloc[:, 1].to_numpy()

    else:
        raw_train = np.load(
            data_dir / f"{dataset_name}_train.npy",
            allow_pickle=False
        )
        raw_test = np.load(
            data_dir / f"{dataset_name}_test.npy",
            allow_pickle=False
        )
        raw_y = np.load(
            data_dir / f"{dataset_name}_test_label.npy",
            allow_pickle=False
        )

        assert raw_y.ndim == 1 or (
            raw_y.ndim == 2 and raw_y.shape[1] == 1
        )
        labels = raw_y.reshape(-1)

    assert raw_train.shape == (
        prep["train_file_rows"], prep["n_features"]
    )
    assert raw_test.shape == (
        prep["test_rows"], prep["n_features"]
    )
    assert labels.shape == (len(raw_test),)
    assert np.isin(labels, [0, 1]).all()
    assert np.isfinite(raw_train).all()
    assert np.isfinite(raw_test).all()

    # Dùng lại scaler đã lưu, không fit một scaler mới.
    scaler = joblib.load(reference / "scaler.joblib")
    assert int(scaler.n_samples_seen_) == len(raw_train)

    train_z = scaler.transform(raw_train).astype(np.float32)
    test_z = scaler.transform(raw_test).astype(np.float32)

    full = PSMWindowDataset(
        train_z, window=60, stride=10
    )
    test_ds = PSMWindowDataset(
        test_z, labels=labels, window=60, stride=10
    )

    with np.load(
        reference / "window_split.npz", allow_pickle=False
    ) as split:
        train_ids = split["train_ids"].copy()
        val_ids = split["val_ids"].copy()

    # Kiểm tra actual split thực sự thuộc seed này.
    all_ids = np.arange(len(full))
    start = random.Random(seed).randrange(
        int(len(full) * 0.8)
    )
    expected_val = all_ids[
        start:start + int(len(full) * 0.2)
    ]

    np.testing.assert_array_equal(val_ids, expected_val)
    np.testing.assert_array_equal(
        train_ids,
        np.r_[all_ids[:start], all_ids[start + len(val_ids):]]
    )
    np.testing.assert_array_equal(
        np.sort(np.r_[train_ids, val_ids]), all_ids
    )

    # Đối chiếu test window độc lập với kết quả MTGFlow.
    truth_start = test_ds.starts
    cumulative = np.r_[0, np.cumsum(labels == 1)]
    truth = (
        cumulative[truth_start + 60]
        - cumulative[truth_start] > 0
    ).astype(int)

    ref_test = pd.read_csv(
        reference / "test_scores.csv"
    ).sort_values("window_id")
    ref_val = pd.read_csv(
        reference / "validation_scores.csv"
    ).sort_values("window_id")

    np.testing.assert_array_equal(
        ref_test["window_id"], np.arange(len(test_ds))
    )
    np.testing.assert_array_equal(ref_test["start"], truth_start)
    np.testing.assert_array_equal(ref_test["label"], truth)
    np.testing.assert_array_equal(
        ref_val["window_id"], np.sort(val_ids)
    )

    # 4. Khởi tạo độc lập, không lấy trọng số MTGFlow.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model_config = {
        "in_channels": prep["n_features"],
        "n_channels": 16,
        "kernel_size": 5
    }

    # Tạo CNNAE như constructor tham chiếu, rồi chỉ dùng encoder.
    ae = SVDD_CNNAE(**model_config).to(device)
    net = ae.encoder
    del ae

    train_loader = DataLoader(
        Subset(full, train_ids.tolist()),
        batch_size=512, shuffle=True,
        drop_last=True, num_workers=0
    )
    val_loader = DataLoader(
        Subset(full, val_ids.tolist()),
        batch_size=512, shuffle=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=512, shuffle=False
    )

    epochs = 20

    config = {
        "model": "DeepSVDD",
        "variant": "mtgflow_repo_cnn_no_pretrain",
        "dataset": dataset_name,
        "seed": seed,
        "epochs": epochs,
        "batch_size": 512,
        "lr": 0.0001,
        "weight_decay": 0.0005,
        "lr_policy": "epoch1_19_1e-4_epoch20_1e-5",
        "model_config": model_config,
        "center_eps": 0.1,
        "center_policy":
            "mean_feature_map_then_eps_clip_then_temporal_mean",
        "score":
            "squared_l2_of_temporally_averaged_encoder_output_to_fixed_c",
        "pretraining_epochs": 0,
        "reference_commit": SVDD_COMMIT,
        "protocol_id": prep["protocol_id"],
        "preprocessing_reference": str(reference),
        "checkpoint_policy": "last_epoch_fixed_in_advance",
        "auprc_definition": "average_precision",
        "threshold_policy": "not_selected",
        "torch": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(0)
    }

    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
    out = root / f"DeepSVDD_{dataset_name}_seed{seed}_{stamp}"
    out.mkdir(parents=True, exist_ok=False)

    for name in ["config.json", "scaler.joblib", "window_split.npz"]:
        shutil.copy2(reference / name, out / name)

    shutil.copytree(SVDD_CACHE, out / "upstream_source")
    (out / "network.py").write_text(
        network_code, encoding="utf-8"
    )
    (out / "run_config.json").write_text(
        json.dumps(config, indent=2), encoding="utf-8"
    )
    (out / "data_manifest.json").write_text(
        json.dumps(data_manifest, indent=2), encoding="utf-8"
    )

    changes = {
        "architecture":
            "Unchanged source ConvEncoder. Width16 is CNNAE constructor "
            "default; caller size is undefined in upstream script, "
            "so exact paper width is not verified.",
        "pretraining":
            "Not called by upstream DeepSVDD entry point; disabled here.",
        "loss_center_score":
            "Reference squared-distance loss and fixed c; "
            "epsilon clipping BEFORE temporal pooling.",
        "adapter":
            "squeeze(-1) input matches source transpose+reshape; "
            "preserve batch axis instead of bare squeeze().",
        "training_mode":
            "Train mode each epoch. Upstream test() sets eval() "
            "without restoring train in following epochs; "
            "no per-epoch test calls here.",
        "scheduler":
            "Explicit LR matching current PyTorch pre-epoch "
            "MultiStepLR milestone20.",
        "evaluation":
            "Last epoch, no test-based selection. Shared DAMR "
            "data/scaler/splits, not original MTGFlow loaders."
    }
    (out / "adapter_notes.json").write_text(
        json.dumps(changes, indent=2), encoding="utf-8"
    )

    print(
        f"DeepSVDD {dataset_name} seed{seed}: "
        f"train/val/test={len(train_ids)}/{len(val_ids)}/{len(test_ds)}"
    )
    print("20 epoch; CNN width16; không AE pretraining")
    print("Dùng split từ:", reference.name)
    print("Output:", out, flush=True)

    # 5. Khởi tạo tâm c theo source.
    optimizer = torch.optim.Adam(
        net.parameters(), lr=1e-4, weight_decay=5e-4
    )

    torch.cuda.synchronize()
    started = time.perf_counter()

    net.eval()
    center_sum, center_count = None, 0

    with torch.no_grad():
        for x, _, _ in train_loader:
            maps = net(x.to(device).squeeze(-1))

            if center_sum is None:
                center_sum = maps.sum(0)
            else:
                center_sum += maps.sum(0)

            center_count += len(x)

    center_map = center_sum / center_count

    # Giữ đúng thứ tự source: clip epsilon trước, rồi mới pool thời gian.
    center_map[
        (center_map.abs() < 0.1) & (center_map < 0)
    ] = -0.1
    center_map[
        (center_map.abs() < 0.1) & (center_map > 0)
    ] = 0.1

    c = center_map.mean(-1).detach()

    assert c.shape == (64,)
    assert torch.isfinite(c).all()
    torch.save(c.cpu(), out / "center_initial.pt")

    # 6. Train DeepSVDD bằng khoảng cách tới tâm c cố định.
    history = []

    for epoch in range(1, epochs + 1):
        net.train()

        # Theo lịch pre-step milestone20 của trainer tham chiếu.
        lr = 1e-4 if epoch < 20 else 1e-5
        for group in optimizer.param_groups:
            group["lr"] = lr

        total, count = 0.0, 0

        for x, _, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)

            z = net(x.to(device).squeeze(-1)).mean(-1)
            loss = ((z - c) ** 2).sum(1).mean()

            assert torch.isfinite(loss).item()
            loss.backward()

            assert all(
                torch.isfinite(p.grad).all().item()
                for p in net.parameters()
                if p.grad is not None
            )

            optimizer.step()

            total += loss.item() * len(x)
            count += len(x)

        history.append({
            "epoch": epoch,
            "train_loss": total / count,
            "lr": lr,
            "trained_windows": count
        })

        tmp = out / "checkpoint_last.tmp"
        torch.save({
            "epoch": epoch,
            "model_state_dict": net.state_dict(),
            "c": c.cpu(),
            "optimizer_state_dict": optimizer.state_dict(),
            "model_config": model_config,
            "run_config": config
        }, tmp)
        tmp.replace(out / "checkpoint_last.pt")

        pd.DataFrame(history).to_csv(
            out / "epoch_history.csv", index=False
        )

        message = (
            f"DeepSVDD {dataset_name} seed{seed}: "
            f"epoch {epoch:02d}/{epochs}, "
            f"loss={total / count:.6f}"
        )
        print(message, flush=True)

        with (out / "train.log").open("a", encoding="utf-8") as f:
            f.write(message + "\n")

    torch.cuda.synchronize()
    train_seconds = time.perf_counter() - started

    # 7. Load checkpoint và c vào model mới.
    eval_started = time.perf_counter()

    saved = torch.load(
        out / "checkpoint_last.pt",
        map_location=device,
        weights_only=True
    )
    assert saved["epoch"] == epochs

    fresh = SVDD_CNNAE(
        **saved["model_config"]
    ).to(device).encoder
    fresh.load_state_dict(
        saved["model_state_dict"], strict=True
    )
    saved_c = saved["c"].to(device)

    net.eval()
    fresh.eval()

    probe = (
        next(iter(test_loader))[0][:8]
        .to(device).squeeze(-1)
    )

    with torch.no_grad():
        torch.testing.assert_close(
            ((net(probe).mean(-1) - c) ** 2).sum(1),
            ((fresh(probe).mean(-1) - saved_c) ** 2).sum(1)
        )

    # 8. Chấm toàn bộ validation/test, lưu score có ID.
    def collect(loader, base, expected_ids, split_name):
        frames = []

        with torch.no_grad():
            for x, y, idx in loader:
                z = fresh(x.to(device).squeeze(-1)).mean(-1)
                scores = ((z - saved_c) ** 2).sum(1)

                assert scores.shape == (len(x),)
                assert torch.isfinite(scores).all()

                ids = idx.numpy()
                frames.append(pd.DataFrame({
                    "split": split_name,
                    "window_id": ids,
                    "start": base.starts[ids],
                    "end_exclusive": base.starts[ids] + 60,
                    "label": y.numpy(),
                    "score": scores.cpu().numpy()
                }))

        frame = (
            pd.concat(frames, ignore_index=True)
            .sort_values("window_id")
            .reset_index(drop=True)
        )

        np.testing.assert_array_equal(
            frame["window_id"], np.sort(expected_ids)
        )
        assert not frame["window_id"].duplicated().any()

        frame.to_csv(
            out / f"{split_name}_scores.csv", index=False
        )
        return frame

    validation = collect(
        val_loader, full, val_ids, "validation"
    )
    test = collect(
        test_loader, test_ds, np.arange(len(test_ds)), "test"
    )

    np.testing.assert_array_equal(test["start"], truth_start)
    np.testing.assert_array_equal(test["label"], truth)

    torch.cuda.synchronize()
    eval_seconds = time.perf_counter() - eval_started

    row = {
        "model": "DeepSVDD",
        "variant": config["variant"],
        "dataset": dataset_name,
        "seed": seed,
        "noise": 0,
        "epoch": epochs,
        "auroc": float(roc_auc_score(truth, test["score"])),
        "auprc": float(
            average_precision_score(truth, test["score"])
        ),
        "f1": np.nan,
        "flagged_rate": np.nan,
        "runtime_s": train_seconds + eval_seconds
    }

    pd.DataFrame([row]).to_csv(
        out / "metrics.csv", index=False
    )

    (out / "completed.json").write_text(
        json.dumps({
            "status": "completed",
            "checkpoint_policy": "last_epoch",
            "validation_windows": len(validation),
            "test_windows": len(test),
            "train_runtime_s": train_seconds,
            "evaluation_runtime_s": eval_seconds,
            "checkpoint_reload_matches": True,
            "f1_status": "not_computed_threshold_not_selected"
        }, indent=2),
        encoding="utf-8"
    )

    print(pd.DataFrame([row]).to_string(index=False))
    print("HOÀN THÀNH:", out)

    del net, fresh, optimizer, saved, probe, maps, z, loss
    gc.collect()
    torch.cuda.empty_cache()

    return out

In [ ]:
SVDD_PSM16_RUN = run_deepsvdd("PSM", 16)

DeepSVDD PSM seed16: train/val/test=10595/2648/8779
20 epoch; CNN width16; không AE pretraining
Dùng split từ: MTGFlow_PSM_seed16_20261122T164705_499086Z
Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/DeepSVDD_PSM_seed16_20261122T183229_256252Z
DeepSVDD PSM seed16: epoch 01/20, loss=17.058445
DeepSVDD PSM seed16: epoch 02/20, loss=13.579121
DeepSVDD PSM seed16: epoch 03/20, loss=10.847937
DeepSVDD PSM seed16: epoch 04/20, loss=8.072024
DeepSVDD PSM seed16: epoch 05/20, loss=6.056066
DeepSVDD PSM seed16: epoch 06/20, loss=4.586005
DeepSVDD PSM seed16: epoch 07/20, loss=3.478577
DeepSVDD PSM seed16: epoch 08/20, loss=2.559311
DeepSVDD PSM seed16: epoch 09/20, loss=2.204952
DeepSVDD PSM seed16: epoch 10/20, loss=1.914702
DeepSVDD PSM seed16: epoch 11/20, loss=1.687017
DeepSVDD PSM seed16: epoch 12/20, loss=1.511273
DeepSVDD PSM seed16: epoch 13/20, loss=1.240062
DeepSVDD PSM seed16: epoch 14/20, loss=1.482037
DeepSVDD PSM seed16: epoch 15/20, loss=1.054519
DeepSVDD PSM seed16:

In [ ]:
assert (Path(SVDD_PSM16_RUN) / "completed.json").is_file()

svdd_remaining_jobs = [
    ("PSM", 17),
    ("PSM", 18),
    ("MSL", 16),
    ("MSL", 17),
    ("MSL", 18),
    ("SMD", 16),
    ("SMD", 17),
    ("SMD", 18),
]

svdd_remaining_folders = []

for dataset_name, seed in svdd_remaining_jobs:
    folder = run_deepsvdd(dataset_name, seed)
    svdd_remaining_folders.append(folder)

print("HOÀN THÀNH TÁM LƯỢT DEEPSVDD CÒN LẠI:")
for folder in svdd_remaining_folders:
    print(folder)

DeepSVDD PSM seed17: train/val/test=10595/2648/8779
20 epoch; CNN width16; không AE pretraining
Dùng split từ: MTGFlow_PSM_seed17_20261122T172442_322830Z
Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/DeepSVDD_PSM_seed17_20261122T183325_031329Z
DeepSVDD PSM seed17: epoch 01/20, loss=17.546094
DeepSVDD PSM seed17: epoch 02/20, loss=13.624987
DeepSVDD PSM seed17: epoch 03/20, loss=10.392006
DeepSVDD PSM seed17: epoch 04/20, loss=7.682284
DeepSVDD PSM seed17: epoch 05/20, loss=5.717263
DeepSVDD PSM seed17: epoch 06/20, loss=4.468055
DeepSVDD PSM seed17: epoch 07/20, loss=3.754305
DeepSVDD PSM seed17: epoch 08/20, loss=3.309120
DeepSVDD PSM seed17: epoch 09/20, loss=2.976864
DeepSVDD PSM seed17: epoch 10/20, loss=2.728654
DeepSVDD PSM seed17: epoch 11/20, loss=2.546940
DeepSVDD PSM seed17: epoch 12/20, loss=2.387936
DeepSVDD PSM seed17: epoch 13/20, loss=2.238360
DeepSVDD PSM seed17: epoch 14/20, loss=2.095681
DeepSVDD PSM seed17: epoch 15/20, loss=1.970725
DeepSVDD PSM seed17:

## 18. USAD — StandardScaler chung, PSM/MSL/SMD × seed 16/17/18

In [ ]:
import ast
import gc
import hashlib
import json
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import DataLoader, Subset

assert Path("/content/drive/MyDrive").is_dir(), "Mount Drive trước."

USAD_COMMIT = "e25af45c8e1c32783aed94fc7e5ab85effef2b1a"

USAD_CACHE = (
    Path("/content/drive/MyDrive/CS313_Nhom5/source_cache")
    / ("USAD_" + USAD_COMMIT)
)
USAD_CACHE.mkdir(parents=True, exist_ok=True)

USAD_HASHES = {
    "usad.py":
        "eea96f59ff0e3d4b2b7de31b8b7d4dbe4c02b104e76fdc04fb2cf5575ca1a444",
    "USAD.ipynb":
        "a656a963a1fae67402f43c39b07abd4cdda28a414394bae38b63dce9930d477f",
    "LICENSE":
        "bc835a2b34b2961d2bfc449e9307745d85ba2ce9d774f4bf129238c5d1aa7fa7",
}

for filename, expected in USAD_HASHES.items():
    path = USAD_CACHE / filename

    if not path.exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/manigalati/usad/"
            f"{USAD_COMMIT}/{filename}",
            timeout=(30, 120)
        )
        response.raise_for_status()
        assert hashlib.sha256(response.content).hexdigest() == expected
        path.write_bytes(response.content)

    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected

# Chỉ nạp ba lớp mạng, không chạy notebook/script gốc.
usad_source = (USAD_CACHE / "usad.py").read_text(encoding="utf-8")

usad_nodes = [
    node for node in ast.parse(usad_source).body
    if isinstance(node, ast.ClassDef)
    and node.name in {"Encoder", "Decoder", "UsadModel"}
]
assert {node.name for node in usad_nodes} == {
    "Encoder", "Decoder", "UsadModel"
}

USAD_NETWORK_CODE = (
    "import torch\nimport torch.nn as nn\n\n"
    + "\n\n".join(ast.unparse(n) for n in usad_nodes)
    + "\n"
)

USAD_NS = {}
exec(
    compile(USAD_NETWORK_CODE, "usad_network.py", "exec"),
    USAD_NS
)
USAD_CLASS = USAD_NS["UsadModel"]


def load_usad_shared_data(dataset_name, seed):
    assert dataset_name in ("PSM", "MSL", "SMD")
    assert seed in (16, 17, 18)
    assert "PSMWindowDataset" in globals(), (
        "Thiếu lớp Dataset ở mục 8."
    )

    root = Path(
        "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
    )
    data_dir = (
        Path("/content/drive/MyDrive/CS313_Nhom5/data")
        / dataset_name
    )

    required = [
        "completed.json", "run_config.json", "config.json",
        "scaler.joblib", "window_split.npz",
        "validation_scores.csv", "test_scores.csv"
    ]

    # Lượt hoàn tất hợp lệ mới nhất, không chọn theo AUROC.
    reference = None

    for folder in sorted(
        root.glob(f"MTGFlow_{dataset_name}_seed{seed}_*"),
        reverse=True
    ):
        if not all((folder / name).is_file() for name in required):
            continue

        cfg = json.loads((folder / "run_config.json").read_text())
        status = json.loads((folder / "completed.json").read_text())

        if (
            status.get("status") == "completed"
            and cfg.get("model") == "MTGFlow"
            and cfg.get("dataset") == dataset_name
            and cfg.get("seed") == seed
            and cfg.get("epochs") == 40
            and cfg.get("checkpoint_policy")
                == "last_epoch_fixed_in_advance"
            and cfg.get("protocol_id")
                == f"DAMR_public_code_{dataset_name}_v1"
        ):
            reference = folder
            break

    assert reference is not None, (
        f"Thiếu MTGFlow hoàn tất: {dataset_name}, seed{seed}"
    )

    prep = json.loads((reference / "config.json").read_text())

    assert prep["dataset"] == dataset_name
    assert prep["seed"] == seed
    assert prep["window"] == 60 and prep["stride"] == 10
    assert prep["val_ratio"] == 0.2
    assert prep["scaler_fit_on"] == (
        "full_train_file_before_window_split"
    )

    if dataset_name == "PSM":
        filenames = ["train.csv", "test.csv", "test_label.csv"]
        meta_name = "PSM_EXPECTED_SHA"
    else:
        filenames = [
            f"{dataset_name}_{s}.npy"
            for s in ["train", "test", "test_label"]
        ]
        meta_name = "EXTRA_DATA_REFERENCES"

    assert meta_name in globals(), (
        f"Thiếu {meta_name}: chạy ô khai báo nguồn dữ liệu trước."
    )

    manifest = {}

    for filename in filenames:
        path = data_dir / filename

        if dataset_name == "PSM":
            expected = PSM_EXPECTED_SHA[filename]
        else:
            expected = EXTRA_DATA_REFERENCES[
                f"{dataset_name}/{filename}"
            ]["sha256"]

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024), b""
            ):
                digest.update(block)

        assert digest.hexdigest() == expected, path
        manifest[filename] = {
            "path": str(path), "sha256": expected
        }

    if dataset_name == "PSM":
        tr = pd.read_csv(data_dir / "train.csv")
        te = pd.read_csv(data_dir / "test.csv")
        lab = pd.read_csv(data_dir / "test_label.csv")

        assert "timestamp" in str(tr.columns[0]).lower()
        assert lab.shape[1] == 2
        assert tr.columns.tolist() == te.columns.tolist()

        np.testing.assert_array_equal(
            te.iloc[:, 0], lab.iloc[:, 0]
        )

        raw_train = np.nan_to_num(tr.values[:, 1:])
        raw_test = np.nan_to_num(te.values[:, 1:])
        labels = lab.iloc[:, 1].to_numpy()

    else:
        raw_train = np.load(
            data_dir / f"{dataset_name}_train.npy",
            allow_pickle=False
        )
        raw_test = np.load(
            data_dir / f"{dataset_name}_test.npy",
            allow_pickle=False
        )
        raw_labels = np.load(
            data_dir / f"{dataset_name}_test_label.npy",
            allow_pickle=False
        )

        assert raw_labels.ndim == 1 or (
            raw_labels.ndim == 2 and raw_labels.shape[1] == 1
        )
        labels = raw_labels.reshape(-1)

    assert raw_train.shape == (
        prep["train_file_rows"], prep["n_features"]
    )
    assert raw_test.shape == (
        prep["test_rows"], prep["n_features"]
    )
    assert labels.shape == (len(raw_test),)
    assert np.isin(labels, [0, 1]).all()
    assert np.isfinite(raw_train).all()
    assert np.isfinite(raw_test).all()

    # Dùng chính scaler đã lưu, không fit mới hoặc chuyển MinMax.
    scaler = joblib.load(reference / "scaler.joblib")

    assert scaler.__class__.__name__ == "StandardScaler"
    assert int(scaler.n_samples_seen_) == len(raw_train)

    train_z = scaler.transform(raw_train).astype(np.float32)
    test_z = scaler.transform(raw_test).astype(np.float32)

    assert np.isfinite(train_z).all()
    assert np.isfinite(test_z).all()

    full = PSMWindowDataset(
        train_z, window=60, stride=10
    )
    test_ds = PSMWindowDataset(
        test_z, labels=labels, window=60, stride=10
    )

    with np.load(
        reference / "window_split.npz", allow_pickle=False
    ) as split:
        train_ids = split["train_ids"].copy()
        val_ids = split["val_ids"].copy()

    ids = np.arange(len(full))
    start = random.Random(seed).randrange(
        int(len(full) * 0.8)
    )

    np.testing.assert_array_equal(
        val_ids,
        ids[start:start + int(len(full) * 0.2)]
    )
    np.testing.assert_array_equal(
        train_ids,
        np.r_[ids[:start], ids[start + len(val_ids):]]
    )
    np.testing.assert_array_equal(
        np.sort(np.r_[train_ids, val_ids]), ids
    )

    starts = test_ds.starts
    cumulative = np.r_[0, np.cumsum(labels == 1)]
    truth = (
        cumulative[starts + 60] - cumulative[starts] > 0
    ).astype(int)

    ref_test = pd.read_csv(
        reference / "test_scores.csv"
    ).sort_values("window_id")
    ref_val = pd.read_csv(
        reference / "validation_scores.csv"
    ).sort_values("window_id")

    np.testing.assert_array_equal(
        ref_test["window_id"], np.arange(len(test_ds))
    )
    np.testing.assert_array_equal(ref_test["start"], starts)
    np.testing.assert_array_equal(ref_test["label"], truth)
    np.testing.assert_array_equal(
        ref_val["window_id"], np.sort(val_ids)
    )

    # Chỉ thống kê hạn chế miền giá trị; không dùng để chỉnh test.
    ranges = {
        name: {
            "below_zero_fraction": float((x < 0).mean()),
            "above_one_fraction": float((x > 1).mean())
        }
        for name, x in [
            ("full_train", train_z), ("test", test_z)
        ]
    }

    return (
        reference, prep, full, test_ds,
        train_ids, val_ids, truth, manifest, ranges
    )


def usad_flatten(x):
    # Dataset: (B, feature, window, 1)
    # USAD notebook: flatten theo (B, window, feature).
    return x.squeeze(-1).transpose(1, 2).reshape(len(x), -1)


def usad_two_steps(model, batch, n, optimizer1, optimizer2):
    # Giữ đúng thứ tự step/zero_grad của source tham chiếu.
    loss1, _ = model.training_step(batch, n)
    assert torch.isfinite(loss1).item()

    loss1.backward()
    assert all(
        torch.isfinite(p.grad).all().item()
        for p in model.parameters()
        if p.grad is not None
    )

    optimizer1.step()
    optimizer1.zero_grad()

    # Phải forward lại sau khi AE1 cập nhật.
    _, loss2 = model.training_step(batch, n)
    assert torch.isfinite(loss2).item()

    loss2.backward()
    assert all(
        torch.isfinite(p.grad).all().item()
        for p in model.parameters()
        if p.grad is not None
    )

    optimizer2.step()
    optimizer2.zero_grad()

    return loss1.item(), loss2.item()


def usad_window_score(model, batch):
    w1 = model.decoder1(model.encoder(batch))
    w3 = model.decoder2(model.encoder(w1))

    return (
        0.5 * ((batch - w1) ** 2).mean(1)
        + 0.5 * ((batch - w3) ** 2).mean(1)
    )


print(
    "Đã nạp USAD: giữ Sigmoid/loss gốc; "
    "dùng StandardScaler chung. Chưa train."
)

Đã nạp USAD: giữ Sigmoid/loss gốc; dùng StandardScaler chung. Chưa train.


## Ô này chỉ định nghĩa hàm.

In [ ]:
def run_usad(dataset_name, seed):
    assert torch.cuda.is_available(), "Chưa có CUDA."
    device = torch.device("cuda:0")

    (
        reference, prep, full, test_ds,
        train_ids, val_ids, truth, manifest, ranges
    ) = load_usad_shared_data(dataset_name, seed)

    # Giá trị từ ví dụ notebook USAD tham chiếu.
    # Window/scaler/split được thay bằng protocol chung của nhóm.
    epochs, batch_size, hidden_size = 100, 7919, 100

    model_config = {
        "w_size": 60 * prep["n_features"],
        "z_size": 60 * hidden_size
    }

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = USAD_CLASS(**model_config).to(device)

    optimizer1 = torch.optim.Adam(
        list(model.encoder.parameters())
        + list(model.decoder1.parameters()),
        lr=0.001,
        weight_decay=0
    )
    optimizer2 = torch.optim.Adam(
        list(model.encoder.parameters())
        + list(model.decoder2.parameters()),
        lr=0.001,
        weight_decay=0
    )

    # Theo ví dụ USAD: không shuffle, không bỏ batch cuối.
    train_loader = DataLoader(
        Subset(full, train_ids.tolist()),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0
    )
    val_loader = DataLoader(
        Subset(full, val_ids.tolist()),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    config = {
        "model": "USAD",
        "variant": "shared_standard_sigmoid",
        "dataset": dataset_name,
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "hidden_size": hidden_size,
        "model_config": model_config,
        "lr": 0.001,
        "weight_decay": 0,
        "alpha": 0.5,
        "beta": 0.5,
        "shuffle": False,
        "drop_last": False,
        "protocol_id": prep["protocol_id"],
        "preprocessing_reference": str(reference),
        "scaler": "StandardScaler",
        "decoder_activation": "Sigmoid_unchanged",
        "checkpoint_policy": "last_epoch_fixed_in_advance",
        "save_every_epochs": 10,
        "also_save_epoch1": True,
        "checkpoint_contents":
            "inference_only_no_optimizer_state_not_training_resume",
        "auprc_definition": "average_precision",
        "threshold_policy": "not_selected",
        "source_commit": USAD_COMMIT,
        "source_sha256": USAD_HASHES,
        "config_source":
            "USAD.ipynb SWaT example; not verified USAD "
            "configuration from DAMR paper",
        "runtime_definition":
            "training_including_epoch_validation_and_checkpoint_IO"
            "_plus_final_evaluation_excluding_preparation",
        "torch": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(0)
    }

    root = Path(
        "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
    )
    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    out = root / f"USAD_{dataset_name}_seed{seed}_{stamp}"
    out.mkdir(parents=True, exist_ok=False)

    for name in ["config.json", "scaler.joblib", "window_split.npz"]:
        shutil.copy2(reference / name, out / name)

    shutil.copytree(USAD_CACHE, out / "upstream_source")
    (out / "network.py").write_text(
        USAD_NETWORK_CODE, encoding="utf-8"
    )

    notes = {
        "architecture_loss_score":
            "Original Encoder/Decoder/UsadModel and training_step. "
            "Original Sigmoid; alpha=beta=.5 score.",
        "optimizer_order":
            "Preserve upstream two Adam steps and zero_grad order, "
            "including cross-decoder residual gradients. "
            "Recompute losses after AE1 step.",
        "preprocessing":
            "StandardScaler to match group DAMR protocol, "
            "not original USAD MinMaxScaler. No clipping. "
            "Sigmoid cannot reconstruct values outside [0,1].",
        "data":
            "Same pinned files, scaler and actual split as MTGFlow. "
            "Group window60/stride10 instead of source window12/stride1. "
            "Merged-sequence boundary limitation unchanged.",
        "evaluation":
            "Validation losses do not use labels or select checkpoint. "
            "Test only after fixed final epoch.",
        "checkpoint":
            "Inference-only model at epoch1, every10 epochs and final. "
            "No automatic training resume."
    }

    for name, value in [
        ("run_config.json", config),
        ("adapter_notes.json", notes),
        ("data_manifest.json", manifest),
        ("range_diagnostics.json", ranges)
    ]:
        (out / name).write_text(
            json.dumps(value, indent=2),
            encoding="utf-8"
        )

    print(
        f"USAD {dataset_name} seed{seed}: "
        f"train/val/test={len(train_ids)}/{len(val_ids)}/{len(test_ds)}"
    )
    print(
        f"{epochs} epoch; batch={batch_size}; "
        f"latent={model_config['z_size']}; StandardScaler + Sigmoid"
    )
    print("Input range diagnostics:", ranges)
    print("Output:", out, flush=True)

    history = []
    torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        total1, total2, count = 0.0, 0.0, 0

        for x, _, _ in train_loader:
            batch = usad_flatten(x).to(device)

            loss1, loss2 = usad_two_steps(
                model, batch, epoch, optimizer1, optimizer2
            )

            total1 += loss1 * len(x)
            total2 += loss2 * len(x)
            count += len(x)

        model.eval()

        with torch.no_grad():
            outputs = [
                model.validation_step(
                    usad_flatten(x).to(device), epoch
                )
                for x, _, _ in val_loader
            ]
            validation_loss = model.validation_epoch_end(outputs)

        assert all(
            np.isfinite(v) for v in validation_loss.values()
        )

        history.append({
            "epoch": epoch,
            "train_loss1": total1 / count,
            "train_loss2": total2 / count,
            "trained_windows": count,
            **validation_loss
        })

        # Model USAD lớn hơn: hạn chế ghi checkpoint lớn mỗi epoch.
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            tmp = out / "checkpoint_last.tmp"

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "model_config": model_config,
                "run_config": config
            }, tmp)

            tmp.replace(out / "checkpoint_last.pt")

        pd.DataFrame(history).to_csv(
            out / "epoch_history.csv", index=False
        )

        message = (
            f"USAD {dataset_name} seed{seed}: "
            f"epoch {epoch:03d}/{epochs} | "
            f"L1={total1 / count:.6f} "
            f"L2={total2 / count:.6f}"
        )
        print(message, flush=True)

        with (out / "train.log").open("a", encoding="utf-8") as f:
            f.write(message + "\n")

    torch.cuda.synchronize()
    train_seconds = time.perf_counter() - started

    # Giải phóng optimizer trước khi nạp model kiểm tra.
    del optimizer1, optimizer2
    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()

    eval_started = time.perf_counter()

    saved = torch.load(
        out / "checkpoint_last.pt",
        map_location="cpu",
        weights_only=True
    )
    assert saved["epoch"] == epochs

    restored = USAD_CLASS(
        **saved["model_config"]
    ).to(device)

    restored.load_state_dict(
        saved["model_state_dict"], strict=True
    )
    restored.eval()
    model.eval()

    probe = usad_flatten(
        next(iter(test_loader))[0][:8]
    ).to(device)

    with torch.no_grad():
        torch.testing.assert_close(
            usad_window_score(model, probe),
            usad_window_score(restored, probe)
        )

    def collect(loader, base, expected_ids, split_name):
        frames = []

        with torch.no_grad():
            for x, y, idx in loader:
                scores = usad_window_score(
                    restored, usad_flatten(x).to(device)
                )

                assert scores.shape == (len(x),)
                assert torch.isfinite(scores).all()

                ids = idx.numpy()

                frames.append(pd.DataFrame({
                    "split": split_name,
                    "window_id": ids,
                    "start": base.starts[ids],
                    "end_exclusive": base.starts[ids] + 60,
                    "label": y.numpy(),
                    "score": scores.cpu().numpy()
                }))

        frame = (
            pd.concat(frames, ignore_index=True)
            .sort_values("window_id")
            .reset_index(drop=True)
        )

        np.testing.assert_array_equal(
            frame["window_id"], np.sort(expected_ids)
        )
        assert not frame["window_id"].duplicated().any()

        frame.to_csv(
            out / f"{split_name}_scores.csv",
            index=False
        )
        return frame

    validation = collect(
        val_loader, full, val_ids, "validation"
    )
    test = collect(
        test_loader, test_ds,
        np.arange(len(test_ds)), "test"
    )

    np.testing.assert_array_equal(
        test["start"], test_ds.starts
    )
    np.testing.assert_array_equal(
        test["label"], truth
    )

    torch.cuda.synchronize()
    eval_seconds = time.perf_counter() - eval_started

    row = {
        "model": "USAD",
        "variant": config["variant"],
        "dataset": dataset_name,
        "seed": seed,
        "noise": 0,
        "epoch": epochs,
        "auroc": float(
            roc_auc_score(truth, test["score"])
        ),
        "auprc": float(
            average_precision_score(truth, test["score"])
        ),
        "f1": np.nan,
        "flagged_rate": np.nan,
        "runtime_s": train_seconds + eval_seconds
    }

    pd.DataFrame([row]).to_csv(
        out / "metrics.csv", index=False
    )

    (out / "completed.json").write_text(
        json.dumps({
            "status": "completed",
            "checkpoint_policy": "last_epoch",
            "validation_windows": len(validation),
            "test_windows": len(test),
            "train_runtime_s": train_seconds,
            "evaluation_runtime_s": eval_seconds,
            "checkpoint_reload_matches": True,
            "f1_status": "not_computed_threshold_not_selected"
        }, indent=2),
        encoding="utf-8"
    )

    print(pd.DataFrame([row]).to_string(index=False))
    print("HOÀN THÀNH:", out)

    del model, restored, saved, probe, batch, x
    gc.collect()
    torch.cuda.empty_cache()

    return out

In [ ]:
USAD_PSM16_RUN = run_usad("PSM", 16)

USAD PSM seed16: train/val/test=10595/2648/8779
100 epoch; batch=7919; latent=6000; StandardScaler + Sigmoid
Input range diagnostics: {'full_train': {'below_zero_fraction': 0.5578608253258958, 'above_one_fraction': 0.14588084329073603}, 'test': {'below_zero_fraction': 0.5774365045935269, 'above_one_fraction': 0.12239705832128504}}
Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/USAD_PSM_seed16_20261122T185236_792694Z
USAD PSM seed16: epoch 001/100 | L1=1.254131 L2=1.244117
USAD PSM seed16: epoch 002/100 | L1=1.223552 L2=-0.105119
USAD PSM seed16: epoch 003/100 | L1=1.143140 L2=-0.526704
USAD PSM seed16: epoch 004/100 | L1=1.138401 L2=-0.691767
USAD PSM seed16: epoch 005/100 | L1=1.152933 L2=-0.802412
USAD PSM seed16: epoch 006/100 | L1=1.169681 L2=-0.883413
USAD PSM seed16: epoch 007/100 | L1=1.181308 L2=-0.934471
USAD PSM seed16: epoch 008/100 | L1=1.189127 L2=-0.974303
USAD PSM seed16: epoch 009/100 | L1=1.195207 L2=-1.004582
USAD PSM seed16: epoch 010/100 | L1=1.199885 L2

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path(
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
)
assert ROOT.is_dir(), "Mount Drive và kiểm tra đường dẫn trước."

required = [
    "completed.json", "metrics.csv", "run_config.json",
    "checkpoint_last.pt", "scaler.joblib", "window_split.npz",
    "validation_scores.csv", "test_scores.csv"
]

records = []

for folder in sorted(ROOT.iterdir()):
    if not folder.is_dir():
        continue
    if not folder.name.startswith(
        ("MTGFlow_", "DeepSVDD_", "USAD_")
    ):
        continue

    missing = [
        name for name in required
        if not (folder / name).is_file()
    ]

    if missing:
        print("CHƯA ĐỦ FILE:", folder.name, "| thiếu:", missing)
        continue

    status = json.loads(
        (folder / "completed.json").read_text(encoding="utf-8")
    )
    if status.get("status") != "completed":
        continue

    cfg = json.loads(
        (folder / "run_config.json").read_text(encoding="utf-8")
    )
    metrics = pd.read_csv(folder / "metrics.csv")
    assert len(metrics) == 1, folder

    records.append({
        "model": cfg["model"],
        "variant": metrics.iloc[0]["variant"],
        "dataset": cfg["dataset"],
        "seed": cfg["seed"],
        "epoch": int(metrics.iloc[0]["epoch"]),
        "auroc": metrics.iloc[0]["auroc"],
        "auprc": metrics.iloc[0]["auprc"],
        "run_dir": str(folder),
        "checkpoint": str(folder / "checkpoint_last.pt")
    })

assert records, "Chưa tìm thấy lượt hoàn tất đủ file."

saved_runs = pd.DataFrame(records)
display(saved_runs)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
INDEX_FILE = ROOT / f"DANH_MUC_LUOT_DA_LUU_{stamp}.csv"
saved_runs.to_csv(INDEX_FILE, index=False)

print("Đã lưu danh mục:", INDEX_FILE)
print("Đây là kiểm tra file tồn tại, không phải train/đánh giá lại.")

,model,variant,dataset,seed,epoch,auroc,auprc,run_dir,checkpoint
0,DeepSVDD,mtgflow_repo_cnn_no_pretrain,MSL,16,20,0.543099,0.188131,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
1,DeepSVDD,mtgflow_repo_cnn_no_pretrain,MSL,17,20,0.575946,0.198578,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
2,DeepSVDD,mtgflow_repo_cnn_no_pretrain,MSL,18,20,0.502240,0.172247,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
3,DeepSVDD,mtgflow_repo_cnn_no_pretrain,PSM,16,20,0.505377,0.330866,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
4,DeepSVDD,mtgflow_repo_cnn_no_pretrain,PSM,17,20,0.390315,0.269488,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
5,DeepSVDD,mtgflow_repo_cnn_no_pretrain,PSM,18,20,0.442932,0.293274,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
6,DeepSVDD,mtgflow_repo_cnn_no_pretrain,SMD,16,20,0.755681,0.194817,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
7,DeepSVDD,mtgflow_repo_cnn_no_pretrain,SMD,17,20,0.724465,0.179641,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
8,DeepSVDD,mtgflow_repo_cnn_no_pretrain,SMD,18,20,0.776196,0.219159,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
9,MTGFlow,base,MSL,16,40,0.644740,0.245798,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...


Đã lưu danh mục: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/DANH_MUC_LUOT_DA_LUU_20261122T185715_387922Z.csv
Đây là kiểm tra file tồn tại, không phải train/đánh giá lại.


In [3]:
import ast
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from google.colab import drive, files

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Chưa kết nối GPU T4."

print("Chọn bản notebook đã lưu: NHK_code (9).ipynb")
uploaded = files.upload()
assert len(uploaded) == 1, "Chỉ chọn một file notebook."

filename, content = next(iter(uploaded.items()))
assert filename.endswith(".ipynb")

backup = json.loads(content.decode("utf-8-sig"))

parsed_cells = []
for cell in backup["cells"]:
    if cell["cell_type"] != "code":
        continue

    source_text = "".join(cell.get("source", []))
    try:
        tree = ast.parse(source_text)
    except SyntaxError:
        continue

    parsed_cells.append((source_text, tree))

# 1. Chỉ lấy các dictionary checksum, không chạy ô tải/E1.
wanted_constants = {"PSM_EXPECTED_SHA", "EXTRA_DATA_REFERENCES"}

for source_text, tree in parsed_cells:
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if (
                    isinstance(target, ast.Name)
                    and target.id in wanted_constants
                ):
                    globals()[target.id] = ast.literal_eval(node.value)

assert wanted_constants.issubset(globals()), (
    "Bản notebook thiếu metadata dữ liệu."
)

# 2. Chỉ nạp định nghĩa Dataset.
dataset_node = None
runner_node = None
usad_setup = None

for source_text, tree in parsed_cells:
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            if node.name == "PSMWindowDataset":
                dataset_node = node

        if isinstance(node, ast.FunctionDef):
            if node.name == "run_usad":
                runner_node = node

            if node.name == "load_usad_shared_data":
                if "USAD_COMMIT" in source_text:
                    usad_setup = source_text

assert dataset_node is not None, "Thiếu PSMWindowDataset."
assert runner_node is not None, "Thiếu run_usad."
assert usad_setup is not None, "Thiếu ô nạp source USAD."

exec(
    compile(
        ast.Module(body=[dataset_node], type_ignores=[]),
        "restore_dataset", "exec"
    ),
    globals()
)

# 3. Nạp source/cache và helper USAD — ô này không gọi train.
exec(
    compile(usad_setup, "restore_usad_setup", "exec"),
    globals()
)

# 4. Chỉ định nghĩa runner, không gọi run_usad().
exec(
    compile(
        ast.Module(body=[runner_node], type_ignores=[]),
        "restore_usad_runner", "exec"
    ),
    globals()
)

print("KHÔI PHỤC XONG.")
print("GPU:", torch.cuda.get_device_name(0))
print("Chưa train lại lượt nào.")
print("Giờ chạy ô queue USAD có kiểm tra bỏ qua lượt đã hoàn tất.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chọn bản notebook đã lưu: NHK_code (9).ipynb


Saving NHK_code (9).ipynb to NHK_code (9).ipynb
Đã nạp USAD: giữ Sigmoid/loss gốc; dùng StandardScaler chung. Chưa train.
KHÔI PHỤC XONG.
GPU: Tesla T4
Chưa train lại lượt nào.
Giờ chạy ô queue USAD có kiểm tra bỏ qua lượt đã hoàn tất.


In [4]:
import json
from pathlib import Path
import pandas as pd
import torch

assert Path("/content/drive/MyDrive").is_dir(), "Mount lại Drive trước."
assert torch.cuda.is_available(), "Kết nối GPU T4 trước."

needed = [
    "run_usad", "load_usad_shared_data",
    "USAD_CLASS", "USAD_CACHE", "USAD_HASHES",
    "USAD_NETWORK_CODE", "USAD_COMMIT",
    "usad_flatten", "usad_two_steps", "usad_window_score",
    "PSMWindowDataset", "PSM_EXPECTED_SHA", "EXTRA_DATA_REFERENCES"
]
missing = [name for name in needed if name not in globals()]
assert not missing, (
    f"Phiên hiện tại thiếu: {missing}. "
    "Nạp lại các định nghĩa/import trước, KHÔNG bấm Chạy tất cả."
)

ROOT = Path(
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
)

required_files = [
    "completed.json", "run_config.json", "config.json",
    "checkpoint_last.pt", "scaler.joblib", "window_split.npz",
    "validation_scores.csv", "test_scores.csv", "metrics.csv"
]

features = {"PSM": 25, "MSL": 55, "SMD": 38}


def find_completed_usad(dataset_name, seed):
    expected = {
        "model": "USAD",
        "variant": "shared_standard_sigmoid",
        "dataset": dataset_name,
        "seed": seed,
        "epochs": 100,
        "batch_size": 7919,
        "hidden_size": 100,
        "lr": 0.001,
        "weight_decay": 0,
        "alpha": 0.5,
        "beta": 0.5,
        "scaler": "StandardScaler",
        "decoder_activation": "Sigmoid_unchanged",
        "checkpoint_policy": "last_epoch_fixed_in_advance",
        "source_commit": USAD_COMMIT,
        "model_config": {
            "w_size": 60 * features[dataset_name],
            "z_size": 6000
        }
    }

    for folder in sorted(
        ROOT.glob(f"USAD_{dataset_name}_seed{seed}_*"),
        reverse=True
    ):
        if not all((folder / name).is_file() for name in required_files):
            continue

        status = json.loads(
            (folder / "completed.json").read_text(encoding="utf-8")
        )
        cfg = json.loads(
            (folder / "run_config.json").read_text(encoding="utf-8")
        )

        if status.get("status") != "completed":
            continue
        if not all(cfg.get(key) == value for key, value in expected.items()):
            continue
        if cfg.get("source_sha256") != USAD_HASHES:
            continue

        prep = json.loads(
            (folder / "config.json").read_text(encoding="utf-8")
        )
        assert prep["window"] == 60 and prep["stride"] == 10
        assert prep["dataset"] == dataset_name and prep["seed"] == seed

        metric = pd.read_csv(folder / "metrics.csv")
        assert len(metric) == 1
        assert metric.iloc[0]["dataset"] == dataset_name
        assert int(metric.iloc[0]["seed"]) == seed
        assert int(metric.iloc[0]["epoch"]) == 100

        return folder

    return None


# Gồm cả PSM–seed16 để kiểm tra và tự bỏ qua lượt đã xong.
usad_jobs = [
    (dataset_name, seed)
    for dataset_name in ["PSM", "MSL", "SMD"]
    for seed in [16, 17, 18]
]

usad_selected_folders = []

for dataset_name, seed in usad_jobs:
    folder = find_completed_usad(dataset_name, seed)

    if folder is not None:
        print(
            f"BỎ QUA — đã hoàn tất: {dataset_name}, seed{seed}",
            flush=True
        )
    else:
        print(
            f"\nBẮT ĐẦU USAD: {dataset_name}, seed{seed}",
            flush=True
        )
        folder = run_usad(dataset_name, seed)

        assert (folder / "completed.json").is_file()

    usad_selected_folders.append(folder)

usad_results = pd.concat(
    [
        pd.read_csv(folder / "metrics.csv")
        for folder in usad_selected_folders
    ],
    ignore_index=True
)

assert len(usad_results) == 9
display(usad_results.sort_values(["dataset", "seed"]))

print("HOÀN THÀNH USAD: 3 dataset × 3 seed = 9 lượt.")

BỎ QUA — đã hoàn tất: PSM, seed16

BẮT ĐẦU USAD: PSM, seed17
USAD PSM seed17: train/val/test=10595/2648/8779
100 epoch; batch=7919; latent=6000; StandardScaler + Sigmoid
Input range diagnostics: {'full_train': {'below_zero_fraction': 0.5578608253258958, 'above_one_fraction': 0.14588084329073603}, 'test': {'below_zero_fraction': 0.5774365045935269, 'above_one_fraction': 0.12239705832128504}}
Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/USAD_PSM_seed17_20261123T030000_694522Z
USAD PSM seed17: epoch 001/100 | L1=1.231482 L2=1.221830
USAD PSM seed17: epoch 002/100 | L1=1.152840 L2=-0.151513
USAD PSM seed17: epoch 003/100 | L1=1.075093 L2=-0.509304
USAD PSM seed17: epoch 004/100 | L1=1.096213 L2=-0.693785
USAD PSM seed17: epoch 005/100 | L1=1.117697 L2=-0.800929
USAD PSM seed17: epoch 006/100 | L1=1.135803 L2=-0.871358
USAD PSM seed17: epoch 007/100 | L1=1.149264 L2=-0.921743
USAD PSM seed17: epoch 008/100 | L1=1.157989 L2=-0.959908
USAD PSM seed17: epoch 009/100 | L1=1.164132

,model,variant,dataset,seed,noise,epoch,auroc,auprc,f1,flagged_rate,runtime_s
3,USAD,shared_standard_sigmoid,MSL,16,0,100,0.597332,0.222657,NaN,NaN,153.494798
4,USAD,shared_standard_sigmoid,MSL,17,0,100,0.598620,0.222932,NaN,NaN,150.556165
5,USAD,shared_standard_sigmoid,MSL,18,0,100,0.595904,0.222175,NaN,NaN,152.700306
0,USAD,shared_standard_sigmoid,PSM,16,0,100,0.629248,0.469536,NaN,NaN,124.728558
1,USAD,shared_standard_sigmoid,PSM,17,0,100,0.630789,0.465256,NaN,NaN,101.361357
2,USAD,shared_standard_sigmoid,PSM,18,0,100,0.629967,0.465849,NaN,NaN,104.579292
6,USAD,shared_standard_sigmoid,SMD,16,0,100,0.719842,0.239357,NaN,NaN,889.051268
7,USAD,shared_standard_sigmoid,SMD,17,0,100,0.717699,0.230810,NaN,NaN,867.427380
8,USAD,shared_standard_sigmoid,SMD,18,0,100,0.721542,0.243043,NaN,NaN,893.432369


HOÀN THÀNH USAD: 3 dataset × 3 seed = 9 lượt.


## Ô Code 1 — Nạp file ALOCC, chưa train

In [5]:
from pathlib import Path
import hashlib
import torch
from google.colab import drive, files

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Kết nối GPU T4 trước."

needed = [
    "load_usad_shared_data",
    "PSMWindowDataset",
    "PSM_EXPECTED_SHA",
    "EXTRA_DATA_REFERENCES",
]
missing = [name for name in needed if name not in globals()]

assert not missing, (
    f"Thiếu định nghĩa: {missing}. "
    "Chạy ô khôi phục USAD trước; KHÔNG chạy lại ô train."
)

# Kiểm tra đúng bản adapter đã kiểm thử.
ALOCC_ADAPTER_SHA256 = (
    "2af3974680b9bf6de4d899064f9a7770"
    "b0fc83db4a73c4d52e5f20bb0ae13f0e"
)

adapter = (
    Path("/content/drive/MyDrive/CS313_Nhom5/source_cache")
    / ("ALOCC_adapter_" + ALOCC_ADAPTER_SHA256[:12])
    / "alocc_colab_cells.py"
)

# Chỉ cần upload lần đầu; các lần sau đọc lại từ Drive.
if not adapter.is_file():
    print("Chọn file alocc_colab_cells.py trên máy bạn.")
    uploaded = files.upload()
    assert len(uploaded) == 1, "Chỉ chọn một file."

    filename, content = next(iter(uploaded.items()))
    assert filename.endswith(".py")
    assert hashlib.sha256(content).hexdigest() == ALOCC_ADAPTER_SHA256, (
        "Không đúng bản file đã kiểm thử."
    )

    adapter.parent.mkdir(parents=True, exist_ok=True)
    adapter.write_bytes(content)

content = adapter.read_bytes()
assert hashlib.sha256(content).hexdigest() == ALOCC_ADAPTER_SHA256

# File này chỉ nạp định nghĩa, KHÔNG gọi train.
exec(
    compile(content.decode("utf-8-sig"), str(adapter), "exec"),
    globals()
)

print("Sẵn sàng ALOCC. GPU:", torch.cuda.get_device_name(0))
print("Adapter đã lưu trên Drive:", adapter)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chọn file alocc_colab_cells.py trên máy bạn.


Saving alocc_colab_cells.py to alocc_colab_cells.py
ALOCC đã nạp; chưa train. Dò shape an toàn; width16 là mặc định repo, chưa xác minh paper DAMR.
Sẵn sàng ALOCC. GPU: Tesla T4
Adapter đã lưu trên Drive: /content/drive/MyDrive/CS313_Nhom5/source_cache/ALOCC_adapter_2af3974680b9/alocc_colab_cells.py


## Chạy PSM–seed16 trước

In [6]:
ALOCC_PSM16_RUN = run_alocc_if_missing("PSM", 16)

ALOCC PSM seed16: train/val/test=10595/2648/8779
40 epoch; batch512; width16; score=-D(real). Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/ALOCC_PSM_seed16_20261123T040103_761904Z
ALOCC PSM seed16: epoch 01/40 | D=1.250080 R=1.081539
ALOCC PSM seed16: epoch 02/40 | D=0.842498 R=1.374055
ALOCC PSM seed16: epoch 03/40 | D=0.512841 R=2.391585
ALOCC PSM seed16: epoch 04/40 | D=0.276436 R=3.055018
ALOCC PSM seed16: epoch 05/40 | D=0.226735 R=3.325648
ALOCC PSM seed16: epoch 06/40 | D=0.166378 R=3.610478
ALOCC PSM seed16: epoch 07/40 | D=0.096130 R=3.769500
ALOCC PSM seed16: epoch 08/40 | D=0.073166 R=3.853520
ALOCC PSM seed16: epoch 09/40 | D=0.082263 R=4.155135
ALOCC PSM seed16: epoch 10/40 | D=0.094341 R=4.859580
ALOCC PSM seed16: epoch 11/40 | D=0.098236 R=4.803747
ALOCC PSM seed16: epoch 12/40 | D=0.065455 R=5.241271
ALOCC PSM seed16: epoch 13/40 | D=0.066369 R=5.463581
ALOCC PSM seed16: epoch 14/40 | D=0.036651 R=6.135281
ALOCC PSM seed16: epoch 15/40 | D=0.058627 R=5.788

In [7]:
import json
import pandas as pd
from datetime import datetime, timezone

assert torch.cuda.is_available(), "Cần GPU T4."

# Không cho chạy hàng loạt trước khi lượt đầu hoàn tất.
assert find_completed_alocc("PSM", 16) is not None, (
    "PSM–seed16 chưa hoàn tất hoặc chưa đúng cấu hình."
)

alocc_folders = []

for dataset_name in ["PSM", "MSL", "SMD"]:
    for seed in [16, 17, 18]:
        folder = run_alocc_if_missing(dataset_name, seed)

        status = json.loads(
            (folder / "completed.json").read_text(encoding="utf-8")
        )
        assert status["status"] == "completed"

        alocc_folders.append(folder)

# Tổng hợp kết quả, không train/đánh giá lại.
frames = []

for folder in alocc_folders:
    frame = pd.read_csv(folder / "metrics.csv")
    frame["run_dir"] = str(folder)
    frames.append(frame)

alocc_results = pd.concat(frames, ignore_index=True)
alocc_results = alocc_results.sort_values(
    ["dataset", "seed"]
).reset_index(drop=True)

assert len(alocc_results) == 9
assert not alocc_results.duplicated(["dataset", "seed"]).any()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

results_path = ALOCC_ROOT / f"ALOCC_9_RUNS_{stamp}.csv"
alocc_results.to_csv(results_path, index=False)

# Pandas std mặc định dùng ddof=1: sample standard deviation.
alocc_summary = (
    alocc_results.groupby(["model", "variant", "dataset"])
    .agg(
        n_seeds=("seed", "nunique"),
        auroc_mean=("auroc", "mean"),
        auroc_std=("auroc", "std"),
        auprc_mean=("auprc", "mean"),
        auprc_std=("auprc", "std"),
        runtime_s_mean=("runtime_s", "mean"),
    )
    .reset_index()
)

assert (alocc_summary["n_seeds"] == 3).all()

summary_path = ALOCC_ROOT / f"ALOCC_MEAN_STD_{stamp}.csv"
alocc_summary.to_csv(summary_path, index=False)

display(alocc_results)
display(alocc_summary)

print("HOÀN THÀNH ALOCC: 9/9 lượt.")
print("Kết quả từng seed:", results_path)
print("Mean/std:", summary_path)

BỎ QUA — đã hoàn tất: ALOCC PSM seed16
ALOCC PSM seed17: train/val/test=10595/2648/8779
40 epoch; batch512; width16; score=-D(real). Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/ALOCC_PSM_seed17_20261123T040140_389208Z
ALOCC PSM seed17: epoch 01/40 | D=1.235450 R=0.997136
ALOCC PSM seed17: epoch 02/40 | D=0.957370 R=1.438187
ALOCC PSM seed17: epoch 03/40 | D=0.635504 R=1.724131
ALOCC PSM seed17: epoch 04/40 | D=0.385535 R=2.139048
ALOCC PSM seed17: epoch 05/40 | D=0.236884 R=2.937756
ALOCC PSM seed17: epoch 06/40 | D=0.200396 R=3.125760
ALOCC PSM seed17: epoch 07/40 | D=0.091824 R=3.423297
ALOCC PSM seed17: epoch 08/40 | D=0.088199 R=4.106451
ALOCC PSM seed17: epoch 09/40 | D=0.053431 R=4.390040
ALOCC PSM seed17: epoch 10/40 | D=0.057490 R=4.649551
ALOCC PSM seed17: epoch 11/40 | D=0.064396 R=4.428863
ALOCC PSM seed17: epoch 12/40 | D=0.078440 R=4.341634
ALOCC PSM seed17: epoch 13/40 | D=0.082421 R=5.103361
ALOCC PSM seed17: epoch 14/40 | D=0.064775 R=5.866122
ALOCC PSM s

,model,variant,dataset,seed,noise,epoch,auroc,auprc,f1,flagged_rate,runtime_s,run_dir
0,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,16,0,40,0.466463,0.159433,NaN,NaN,14.300977,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
1,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,17,0,40,0.501979,0.149458,NaN,NaN,13.881529,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
2,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,18,0,40,0.553953,0.201339,NaN,NaN,14.083922,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
3,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,16,0,40,0.544078,0.446976,NaN,NaN,26.224358,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
4,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,17,0,40,0.409175,0.277624,NaN,NaN,26.629587,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
5,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,18,0,40,0.523611,0.384757,NaN,NaN,25.768228,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
6,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,16,0,40,0.463362,0.084711,NaN,NaN,129.885410,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
7,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,17,0,40,0.529242,0.082280,NaN,NaN,128.570541,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...
8,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,18,0,40,0.536392,0.081097,NaN,NaN,129.909863,/content/drive/MyDrive/CS313_Nhom5/hoang_khang...


,model,variant,dataset,n_seeds,auroc_mean,auroc_std,auprc_mean,auprc_std,runtime_s_mean
0,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,3,0.507465,0.044002,0.170076,0.027530,14.088809
1,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,3,0.492288,0.072702,0.369786,0.085663,26.207391
2,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,3,0.509665,0.040259,0.082696,0.001843,129.455272


HOÀN THÀNH ALOCC: 9/9 lượt.
Kết quả từng seed: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/ALOCC_9_RUNS_20261123T040956_053151Z.csv
Mean/std: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/ALOCC_MEAN_STD_20261123T040956_053151Z.csv


## Nạp GANF

In [8]:
from pathlib import Path
import hashlib
import torch
from google.colab import drive, files

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Kết nối GPU T4 trước."

needed = [
    "load_usad_shared_data",
    "PSMWindowDataset",
    "PSM_EXPECTED_SHA",
    "EXTRA_DATA_REFERENCES",
]
missing = [name for name in needed if name not in globals()]

assert not missing, (
    f"Thiếu định nghĩa: {missing}. "
    "Cần khôi phục các hàm dữ liệu chung, KHÔNG train lại."
)

expected_sha = (
    "ea118dffa00eb53f6439f109cf7f9271"
    "69ce7b6f56e9136c43aa161552c1f323"
)

GANF_ADAPTER_PATH = (
    Path("/content/drive/MyDrive/CS313_Nhom5/source_cache")
    / ("GANF_adapter_" + expected_sha[:12])
    / "ganf_colab_cells.py"
)

if not GANF_ADAPTER_PATH.is_file():
    print("Chọn file ganf_colab_cells.py, KHÔNG chọn notebook .ipynb.")
    uploaded = files.upload()
    assert len(uploaded) == 1, "Chỉ chọn một file."

    filename, content = next(iter(uploaded.items()))
    assert filename.endswith(".py")
    assert hashlib.sha256(content).hexdigest() == expected_sha, (
        "Không đúng bản GANF đã kiểm thử."
    )

    GANF_ADAPTER_PATH.parent.mkdir(parents=True, exist_ok=True)
    GANF_ADAPTER_PATH.write_bytes(content)

content = GANF_ADAPTER_PATH.read_bytes()
assert hashlib.sha256(content).hexdigest() == expected_sha

exec(
    compile(
        content.decode("utf-8-sig"),
        str(GANF_ADAPTER_PATH),
        "exec"
    ),
    globals()
)

print("SẴN SÀNG GANF. GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chọn file ganf_colab_cells.py, KHÔNG chọn notebook .ipynb.


Saving ganf_colab_cells.py to ganf_colab_cells.py
GANF đã nạp, CHƯA train. Dùng dữ liệu/scaler/split chung; checkpoint lưu cả model+A.
SẴN SÀNG GANF. GPU: Tesla T4


## Chạy PSM–seed16 trước

In [9]:
GANF_PSM16_RUN = run_ganf_if_missing("PSM", 16)

GANF PSM seed16: train/val/test=10595/2648/8779
Lịch train nguồn: augmented DAG rounds +30 final epochs; batch512. Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/GANF_PSM_seed16_20261123T043132_158162Z
GANF PSM seed16: epoch 001 [augmented] NLL=0.715505 h=0.320442 rho=1
GANF PSM seed16: epoch 002 [augmented] NLL=-0.854763 h=0.154678 rho=1
GANF PSM seed16: epoch 003 [augmented] NLL=-0.868121 h=0.0893536 rho=1
GANF PSM seed16: epoch 004 [augmented] NLL=-1.172623 h=0.0501251 rho=10
GANF PSM seed16: epoch 005 [augmented] NLL=-1.322977 h=0.0282879 rho=10
GANF PSM seed16: epoch 006 [augmented] NLL=-1.636645 h=0.0126781 rho=100
GANF PSM seed16: epoch 007 [augmented] NLL=-1.277360 h=0.00588417 rho=100
GANF PSM seed16: epoch 008 [augmented] NLL=-1.218163 h=0.00260925 rho=100
GANF PSM seed16: epoch 009 [augmented] NLL=-1.524245 h=0.00103378 rho=100
GANF PSM seed16: epoch 010 [augmented] NLL=-1.481048 h=0.000480652 rho=100
GANF PSM seed16: epoch 011 [augmented] NLL=-2.089696 h=0.00032

In [ ]:
import pandas as pd
from datetime import datetime, timezone

assert find_completed_ganf("PSM", 16) is not None, (
    "PSM–seed16 chưa hoàn tất đúng cấu hình."
)

ganf_folders = []

for dataset_name in ["PSM", "MSL", "SMD"]:
    for seed in [16, 17, 18]:
        folder = run_ganf_if_missing(dataset_name, seed)
        ganf_folders.append(folder)

ganf_results = pd.concat(
    [pd.read_csv(folder / "metrics.csv") for folder in ganf_folders],
    ignore_index=True
).sort_values(["dataset", "seed"]).reset_index(drop=True)

assert len(ganf_results) == 9
assert not ganf_results.duplicated(["dataset", "seed"]).any()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
results_file = GANF_ROOT / f"GANF_9_RUNS_{stamp}.csv"
ganf_results.to_csv(results_file, index=False)

display(ganf_results)
print("HOÀN THÀNH GANF: 9/9 lượt.")
print("Đã lưu:", results_file)

BỎ QUA — đã hoàn tất: GANF PSM seed16
GANF PSM seed17: train/val/test=10595/2648/8779
Lịch train nguồn: augmented DAG rounds +30 final epochs; batch512. Output: /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a/GANF_PSM_seed17_20261123T043327_945945Z
GANF PSM seed17: epoch 001 [augmented] NLL=0.084213 h=0.335836 rho=1
GANF PSM seed17: epoch 002 [augmented] NLL=-0.970182 h=0.164223 rho=1
GANF PSM seed17: epoch 003 [augmented] NLL=-1.159336 h=0.0830307 rho=1
GANF PSM seed17: epoch 004 [augmented] NLL=-1.361346 h=0.0350761 rho=10
GANF PSM seed17: epoch 005 [augmented] NLL=-1.338435 h=0.0146484 rho=10
GANF PSM seed17: epoch 006 [augmented] NLL=-1.114722 h=0.00660706 rho=10
GANF PSM seed17: epoch 007 [augmented] NLL=-1.527888 h=0.00293541 rho=10
GANF PSM seed17: epoch 008 [augmented] NLL=-1.990359 h=0.00128365 rho=10
GANF PSM seed17: epoch 009 [augmented] NLL=0.400925 h=0.000488281 rho=10
GANF PSM seed17: epoch 010 [augmented] NLL=-1.419048 h=0.000190735 rho=10
GANF PSM seed17: epoch 011 [

In [ ]:
HANDOVER_DIR, ALL_RESULTS, ALL_SUMMARY = export_nhk_handover()

display(ALL_RESULTS)
display(ALL_SUMMARY)

print("THƯ MỤC BÀN GIAO:", HANDOVER_DIR)